<a href="https://colab.research.google.com/github/Besendnju/Advanced-analysis-/blob/main/machine_learning_flood_analysis_in_the_coast_of_cameroon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# 1. INSTALLATION AND IMPORTS
# ============================================================

!pip -q install geemap earthengine-api xgboost scikit-learn pandas numpy matplotlib seaborn

import ee
import geemap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    brier_score_loss
)

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier

print("Libraries imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 63.6 MB/s eta 0:00:00
Libraries imported successfully.


In [3]:
# ============================================================
# 2. GOOGLE EARTH ENGINE INITIALIZATION
# ============================================================

ee.Authenticate()

ee.Initialize(
    project='weighty-tensor-441119-q2',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

print("Google Earth Engine initialized successfully.")

Google Earth Engine initialized successfully.


In [4]:
# ============================================================
# 3. GLOBAL STUDY PARAMETERS
# ============================================================

STUDY_NAME = "Coastal Cameroon Flood Susceptibility"

# Analysis CRS
ANALYSIS_CRS = 'EPSG:32632'

# Primary modelling resolution
ANALYSIS_SCALE = 30

# Current date
TODAY = '2026-08-09'

# Maximum completed date currently available to us
DATA_CUTOFF = '2026-08-09'

# Sentinel-2 composite period
S2_START = '2021-01-01'
S2_END = '2026-08-09'

# CHIRPS historical period
CHIRPS_START = '1981-01-01'
CHIRPS_END = '2026-08-09'

# GPM period
GPM_START = '1998-01-01'
GPM_END = '2026-08-09'

print("Study:", STUDY_NAME)
print("CRS:", ANALYSIS_CRS)
print("Scale:", ANALYSIS_SCALE, "m")

Study: Coastal Cameroon Flood Susceptibility
CRS: EPSG:32632
Scale: 30 m


In [5]:
# ============================================================
# 4. DEFINITIVE COASTAL CAMEROON STUDY ROI
# ============================================================

STUDY_ROI = ee.Geometry.Polygon(
    [[
        [8.668213, 2.547988],
        [8.668213, 4.861101],
        [10.645752, 4.861101],
        [10.645752, 2.547988],
        [8.668213, 2.547988]
    ]],
    proj='EPSG:4326',
    geodesic=False
)

print("STUDY_ROI successfully created.")

print(
    STUDY_ROI.getInfo()
)

STUDY_ROI successfully created.
{'geodesic': False, 'type': 'Polygon', 'coordinates': [[[8.668213, 2.547988], [8.668213, 4.861101], [10.645752, 4.861101], [10.645752, 2.547988], [8.668213, 2.547988]]]}


In [6]:
# ============================================================
# 5. STUDY MAP
# ============================================================

Map = geemap.Map(
    basemap='SATELLITE'
)

Map.centerObject(
    STUDY_ROI,
    8
)

Map.addLayer(
    STUDY_ROI,
    {
        'color': 'red'
    },
    'Coastal Cameroon ROI'
)

Map

Map(center=[3.704223963390629, 9.656982499999982], controls=(WidgetControl(options=['position', 'transparent_b…

In [7]:
# ============================================================
# 6. ROI AREA
# ============================================================

roi_area_km2 = (
    STUDY_ROI
    .area(
        maxError=100
    )
    .divide(1e6)
)

print(
    "Study area (km²):",
    roi_area_km2.getInfo()
)

Study area (km²): 56432.58556156796


In [8]:
# 7. DATASET AVAILABILITY TEST
# ============================================================

s1_test = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(STUDY_ROI)
    .filterDate(
        '2015-01-01',
        '2026-08-09'
    )
)

s2_test = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(STUDY_ROI)
    .filterDate(
        '2017-01-01',
        '2026-08-09'
    )
)

chirps_test = (
    ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    .filterBounds(STUDY_ROI)
    .filterDate(
        '1981-01-01',
        '2026-08-09'
    )
)

gpm_test = (
    ee.ImageCollection('NASA/GPM_L3/IMERG_V07')
    .filterBounds(STUDY_ROI)
    .filterDate(
        '1998-01-01',
        '2026-08-09'
    )
)

print(
    "Sentinel-1 images:",
    s1_test.size().getInfo()
)

print(
    "Sentinel-2 images:",
    s2_test.size().getInfo()
)

print(
    "CHIRPS images:",
    chirps_test.size().getInfo()
)

print(
    "GPM images:",
    gpm_test.size().getInfo()
)

Sentinel-1 images: 3458
Sentinel-2 images: 10521
CHIRPS images: 16617
GPM images: 501414


In [9]:
# ============================================================
# 8. NASADEM TERRAIN
# ============================================================

NASADEM = (
    ee.Image(
        'NASA/NASADEM_HGT/001'
    )
    .select('elevation')
    .clip(STUDY_ROI)
)

Elevation = (
    NASADEM
    .rename('Elevation')
)


# ------------------------------------------------------------
# Slope
# ------------------------------------------------------------

Slope = (
    ee.Terrain
    .slope(Elevation)
    .rename('Slope')
)


# ------------------------------------------------------------
# Aspect
# ------------------------------------------------------------

Aspect = (
    ee.Terrain
    .aspect(Elevation)
    .rename('Aspect')
)


# ------------------------------------------------------------
# Terrain ruggedness index
# ------------------------------------------------------------

TRI_kernel = ee.Kernel.square(
    radius=1,
    units='pixels',
    normalize=False
)

Elevation_mean = (
    Elevation
    .reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=TRI_kernel
    )
)

TRI = (
    Elevation
    .subtract(Elevation_mean)
    .abs()
    .rename('TRI')
)


# ------------------------------------------------------------
# Curvature
# ------------------------------------------------------------

Curvature = (
    Elevation
    .convolve(
        ee.Kernel.laplacian8(
            normalize=False
        )
    )
    .rename('Curvature')
)


# ------------------------------------------------------------
# TPI
# ------------------------------------------------------------

TPI_kernel = ee.Kernel.circle(
    radius=300,
    units='meters',
    normalize=False
)

TPI_mean = (
    Elevation
    .reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=TPI_kernel
    )
)

TPI = (
    Elevation
    .subtract(TPI_mean)
    .rename('TPI')
)

print("Terrain variables created.")


Terrain variables created.


In [10]:
# ============================================================
# 9. MERIT HYDROGRAPHY
# ============================================================

MERIT = (
    ee.Image(
        'MERIT/Hydro/v1_0_1'
    )
    .clip(STUDY_ROI)
)


# ------------------------------------------------------------
# Upstream drainage area
# ------------------------------------------------------------

FlowAccumulation = (
    MERIT
    .select('upa')
    .rename('FlowAccumulation')
)


# ------------------------------------------------------------
# Log-transformed flow accumulation
# ------------------------------------------------------------

LogFlowAccumulation = (
    FlowAccumulation
    .add(0.001)
    .log10()
    .rename('LogFlowAccumulation')
)


# ------------------------------------------------------------
# HAND
# ------------------------------------------------------------

HAND = (
    MERIT
    .select('hnd')
    .rename('HAND')
)


# ------------------------------------------------------------
# River width
# ------------------------------------------------------------

RiverWidth = (
    MERIT
    .select('wth')
    .rename('RiverWidth')
)


print("MERIT hydrological variables created.")

MERIT hydrological variables created.


In [11]:
# ============================================================
# 10. TWI AND SPI
# ============================================================

SlopeRadians = (
    Slope
    .multiply(
        np.pi / 180.0
    )
)

TanSlope = (
    SlopeRadians
    .tan()
    .max(0.001)
)


# ------------------------------------------------------------
# TWI
# ------------------------------------------------------------

TWI = (
    FlowAccumulation
    .add(1)
    .log()
    .subtract(
        TanSlope.log()
    )
    .rename('TWI')
)


# ------------------------------------------------------------
# SPI
# ------------------------------------------------------------

SPI = (
    FlowAccumulation
    .add(1)
    .multiply(
        TanSlope
    )
    .rename('SPI')
)

print("TWI and SPI created.")

TWI and SPI created.


In [12]:
# ============================================================
# 11. JRC GLOBAL SURFACE WATER
# ============================================================

GSW = (
    ee.Image(
        'JRC/GSW1_4/GlobalSurfaceWater'
    )
    .clip(STUDY_ROI)
)


# ------------------------------------------------------------
# Water occurrence
# ------------------------------------------------------------

WaterOccurrence = (
    GSW
    .select('occurrence')
    .rename('WaterOccurrence')
)


# ------------------------------------------------------------
# Water seasonality
# ------------------------------------------------------------

WaterSeasonality = (
    GSW
    .select('seasonality')
    .rename('WaterSeasonality')
)


# ------------------------------------------------------------
# Permanent water
# ------------------------------------------------------------

PermanentWater = (
    WaterOccurrence
    .gte(80)
    .rename('PermanentWater')
)


print("JRC water variables created.")

JRC water variables created.


In [13]:
# 12. DISTANCE TO PERMANENT WATER
# ============================================================

PermanentWaterMask = (
    PermanentWater
    .selfMask()
)

DistanceToPermanentWater = (
    PermanentWaterMask
    .fastDistanceTransform(
        1024,
        'pixels',
        'squared_euclidean'
    )
    .sqrt()
    .multiply(
        ANALYSIS_SCALE
    )
    .rename(
        'DistanceToPermanentWater'
    )
)

print("Distance to permanent water created.")

Distance to permanent water created.


In [14]:
# ============================================================
# 13. CHIRPS RAINFALL BASELINE
# ============================================================

CHIRPS = (
    ee.ImageCollection(
        'UCSB-CHG/CHIRPS/DAILY'
    )
    .filterBounds(
        STUDY_ROI
    )
    .filterDate(
        CHIRPS_START,
        CHIRPS_END
    )
    .select(
        'precipitation'
    )
)


# ------------------------------------------------------------
# Mean daily precipitation
# ------------------------------------------------------------

MeanPrecipitation = (
    CHIRPS
    .mean()
    .rename(
        'MeanPrecipitation'
    )
    .clip(STUDY_ROI)
)


# ------------------------------------------------------------
# Maximum daily rainfall
# ------------------------------------------------------------

MaxDailyPrecipitation = (
    CHIRPS
    .max()
    .rename(
        'MaxDailyPrecipitation'
    )
    .clip(STUDY_ROI)
)


# ------------------------------------------------------------
# Rainfall standard deviation
# ------------------------------------------------------------

RainfallStdDev = (
    CHIRPS
    .reduce(
        ee.Reducer.stdDev()
    )
    .rename(
        'RainfallStdDev'
    )
    .clip(STUDY_ROI)
)

print("CHIRPS baseline variables created.")

CHIRPS baseline variables created.


In [15]:
# ============================================================
# 14. GPM IMERG
# ============================================================

GPM = (
    ee.ImageCollection(
        'NASA/GPM_L3/IMERG_V07'
    )
    .filterBounds(
        STUDY_ROI
    )
    .filterDate(
        GPM_START,
        GPM_END
    )
    .select(
        'precipitation'
    )
)


# Maximum observed IMERG precipitation rate
# over the study period.

GPM_MaxPrecipitation = (
    GPM
    .max()
    .rename(
        'GPM_MaxPrecipitation'
    )
    .clip(STUDY_ROI)
)

print("GPM variable created.")

GPM variable created.


In [16]:
# ============================================================
# 15. ERA5-LAND SOIL MOISTURE
# ============================================================

ERA5_LAND = (
    ee.ImageCollection(
        'ECMWF/ERA5_LAND/HOURLY'
    )
    .filterBounds(
        STUDY_ROI
    )
    .filterDate(
        '2015-01-01',
        DATA_CUTOFF
    )
)


# ------------------------------------------------------------
# Top-layer soil moisture
# ------------------------------------------------------------

SoilMoisture = (
    ERA5_LAND
    .select(
        'volumetric_soil_water_layer_1'
    )
    .mean()
    .rename(
        'SoilMoisture'
    )
    .clip(STUDY_ROI)
)


# ------------------------------------------------------------
# Surface runoff
# ------------------------------------------------------------

SurfaceRunoff = (
    ERA5_LAND
    .select(
        'surface_runoff'
    )
    .mean()
    .rename(
        'SurfaceRunoff'
    )
    .clip(STUDY_ROI)
)

print("ERA5-Land variables created.")

ERA5-Land variables created.


In [17]:
# ============================================================
# 16. SENTINEL-2 SURFACE REFLECTANCE
# ============================================================

def mask_sentinel2(image):

    scl = image.select(
        'SCL'
    )

    valid = (
        scl.neq(3)      # cloud shadow
        .And(
            scl.neq(8)  # cloud medium probability
        )
        .And(
            scl.neq(9)  # cloud high probability
        )
        .And(
            scl.neq(10) # cirrus
        )
        .And(
            scl.neq(11) # snow/ice
        )
    )

    return (
        image
        .updateMask(valid)
        .divide(10000)
        .copyProperties(
            image,
            image.propertyNames()
        )
    )


S2 = (
    ee.ImageCollection(
        'COPERNICUS/S2_SR_HARMONIZED'
    )
    .filterBounds(
        STUDY_ROI
    )
    .filterDate(
        S2_START,
        S2_END
    )
    .filter(
        ee.Filter.lt(
            'CLOUDY_PIXEL_PERCENTAGE',
            30
        )
    )
    .map(
        mask_sentinel2
    )
)

print(
    "Sentinel-2 images:",
    S2.size().getInfo()
)


S2_Median = (
    S2
    .median()
    .clip(STUDY_ROI)
)

Sentinel-2 images: 658


In [18]:
# ============================================================
# 17. SENTINEL-2 SPECTRAL INDICES
# ============================================================

NDVI = (
    S2_Median
    .normalizedDifference(
        ['B8', 'B4']
    )
    .rename('NDVI')
)


NDWI = (
    S2_Median
    .normalizedDifference(
        ['B3', 'B8']
    )
    .rename('NDWI')
)


MNDWI = (
    S2_Median
    .normalizedDifference(
        ['B3', 'B11']
    )
    .rename('MNDWI')
)


NDBI = (
    S2_Median
    .normalizedDifference(
        ['B11', 'B8']
    )
    .rename('NDBI')
)


EVI = (
    S2_Median.expression(
        '2.5*((NIR-RED)/(NIR+6*RED-7.5*BLUE+1))',
        {
            'NIR':
                S2_Median.select('B8'),

            'RED':
                S2_Median.select('B4'),

            'BLUE':
                S2_Median.select('B2')
        }
    )
    .rename('EVI')
)

print("Sentinel-2 indices created.")

Sentinel-2 indices created.


In [19]:
# ============================================================
# 18. ESA WORLDCOVER
# ============================================================

WorldCover = (
    ee.ImageCollection(
        'ESA/WorldCover/v200'
    )
    .first()
    .select('Map')
    .rename('LULC')
    .clip(STUDY_ROI)
)

print("WorldCover loaded.")

WorldCover loaded.


In [20]:
# ============================================================
# 19. DYNAMIC WORLD
# ============================================================

DW = (
    ee.ImageCollection(
        'GOOGLE/DYNAMICWORLD/V1'
    )
    .filterBounds(
        STUDY_ROI
    )
    .filterDate(
        '2021-01-01',
        DATA_CUTOFF
    )
)


DW_WaterProbability = (
    DW
    .select('water')
    .mean()
    .rename(
        'DW_WaterProbability'
    )
    .clip(STUDY_ROI)
)


DW_BuiltProbability = (
    DW
    .select('built')
    .mean()
    .rename(
        'DW_BuiltProbability'
    )
    .clip(STUDY_ROI)
)


DW_VegetationProbability = (
    DW
    .select('vegetation')
    .mean()
    .rename(
        'DW_VegetationProbability'
    )
    .clip(STUDY_ROI)
)

print("Dynamic World variables created.")

Dynamic World variables created.


In [21]:
# ============================================================
# 20. SOILGRIDS WATER CONTENT
# ============================================================

SoilWater = (
    ee.Image(
        'ISRIC/SoilGrids250m/v2_0/wv0010'
    )
    .select(
        'val_0_5cm_Q0_50'
    )
    .rename(
        'SoilWaterContent'
    )
    .clip(STUDY_ROI)
)

print("Soil water content loaded.")

Soil water content loaded.


In [22]:
# ============================================================
# 21. GHSL BUILT-UP SURFACE
# ============================================================

GHSL_BUILT = (
    ee.Image(
        'JRC/GHSL/P2023A/GHS_BUILT_S/2020'
    )
    .select(
        'built_surface'
    )
    .rename(
        'BuiltSurface'
    )
    .clip(STUDY_ROI)
)

print("GHSL built-up surface loaded.")

GHSL built-up surface loaded.


In [23]:
# ============================================================
# 22. GHSL POPULATION — EXPOSURE ONLY
# ============================================================

GHSL_POPULATION = (
    ee.Image(
        'JRC/GHSL/P2023A/GHS_POP/2025'
    )
    .select(
        'population_count'
    )
    .rename(
        'Population'
    )
    .clip(STUDY_ROI)
)

print("GHSL population exposure layer loaded.")

GHSL population exposure layer loaded.


In [24]:
# ============================================================
# 23. FLOOD EVENT CATALOGUE
# ============================================================

EVENTS = [

    {
        'event_id': 'EVENT_001',
        'pre_start': '2015-06-01',
        'pre_end':   '2015-06-18',
        'post_start':'2015-06-22',
        'post_end':  '2015-07-02'
    },

    {
        'event_id': 'EVENT_002',
        'pre_start': '2015-06-20',
        'pre_end':   '2015-07-02',
        'post_start':'2015-07-04',
        'post_end':  '2015-07-15'
    },

    {
        'event_id': 'EVENT_003',
        'pre_start': '2016-06-01',
        'pre_end':   '2016-06-20',
        'post_start':'2016-06-23',
        'post_end':  '2016-07-05'
    },

    {
        'event_id': 'EVENT_004',
        'pre_start': '2017-08-25',
        'pre_end':   '2017-09-05',
        'post_start':'2017-09-08',
        'post_end':  '2017-09-20'
    },

    {
        'event_id': 'EVENT_005',
        'pre_start': '2018-07-10',
        'pre_end':   '2018-07-23',
        'post_start':'2018-07-26',
        'post_end':  '2018-08-05'
    },

    {
        'event_id': 'EVENT_006',
        'pre_start': '2022-09-01',
        'pre_end':   '2022-09-17',
        'post_start':'2022-09-20',
        'post_end':  '2022-10-02'
    },

    {
        'event_id': 'EVENT_007',
        'pre_start': '2023-07-05',
        'pre_end':   '2023-07-20',
        'post_start':'2023-07-23',
        'post_end':  '2023-08-05'
    },

    {
        'event_id': 'EVENT_008_AUG2026',
        'pre_start': '2026-07-20',
        'pre_end':   '2026-08-03',
        'post_start':'2026-08-04',
        'post_end':  '2026-08-10'
    }
]

print(
    "Number of catalogue events:",
    len(EVENTS)
)

Number of catalogue events: 8


In [25]:
# ============================================================
# 24. SENTINEL-1 EVENT AVAILABILITY VALIDATION
# ============================================================

def check_event(event):

    pre_collection = (
        ee.ImageCollection(
            'COPERNICUS/S1_GRD'
        )
        .filterBounds(
            STUDY_ROI
        )
        .filterDate(
            event['pre_start'],
            event['pre_end']
        )
        .filter(
            ee.Filter.eq(
                'instrumentMode',
                'IW'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VV'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VH'
            )
        )
    )

    post_collection = (
        ee.ImageCollection(
            'COPERNICUS/S1_GRD'
        )
        .filterBounds(
            STUDY_ROI
        )
        .filterDate(
            event['post_start'],
            event['post_end']
        )
        .filter(
            ee.Filter.eq(
                'instrumentMode',
                'IW'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VV'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VH'
            )
        )
    )

    return {
        'event_id':
            event['event_id'],

        'pre_images':
            pre_collection.size().getInfo(),

        'post_images':
            post_collection.size().getInfo(),

        'pre_start':
            event['pre_start'],

        'pre_end':
            event['pre_end'],

        'post_start':
            event['post_start'],

        'post_end':
            event['post_end']
    }


event_status = []

for event in EVENTS:

    status = check_event(event)

    event_status.append(status)

event_status_df = pd.DataFrame(
    event_status
)

display(
    event_status_df
)

,event_id,pre_images,post_images,pre_start,pre_end,post_start,post_end
0,EVENT_001,7,6,2015-06-01,2015-06-18,2015-06-22,2015-07-02
1,EVENT_002,6,3,2015-06-20,2015-07-02,2015-07-04,2015-07-15
2,EVENT_003,6,5,2016-06-01,2016-06-20,2016-06-23,2016-07-05
3,EVENT_004,10,10,2017-08-25,2017-09-05,2017-09-08,2017-09-20
4,EVENT_005,11,6,2018-07-10,2018-07-23,2018-07-26,2018-08-05
5,EVENT_006,13,8,2022-09-01,2022-09-17,2022-09-20,2022-10-02
6,EVENT_007,14,16,2023-07-05,2023-07-20,2023-07-23,2023-08-05
7,EVENT_008_AUG2026,23,7,2026-07-20,2026-08-03,2026-08-04,2026-08-10


In [26]:
# ============================================================
# 25. VALID FLOOD EVENTS
# ============================================================

VALID_EVENTS = []

for event, status in zip(
    EVENTS,
    event_status
):

    # Exclude future/incomplete event
    # and events without both pre and post imagery.

    if (
        status['pre_images'] > 0
        and
        status['post_images'] > 0
        and
        event['post_end'] <= TODAY
    ):

        VALID_EVENTS.append(
            event
        )

print(
    "Valid completed events:",
    len(VALID_EVENTS)
)

for event in VALID_EVENTS:

    print(
        event['event_id'],
        "|",
        event['pre_start'],
        "→",
        event['pre_end'],
        "|",
        event['post_start'],
        "→",
        event['post_end']
    )

Valid completed events: 7
EVENT_001 | 2015-06-01 → 2015-06-18 | 2015-06-22 → 2015-07-02
EVENT_002 | 2015-06-20 → 2015-07-02 | 2015-07-04 → 2015-07-15
EVENT_003 | 2016-06-01 → 2016-06-20 | 2016-06-23 → 2016-07-05
EVENT_004 | 2017-08-25 → 2017-09-05 | 2017-09-08 → 2017-09-20
EVENT_005 | 2018-07-10 → 2018-07-23 | 2018-07-26 → 2018-08-05
EVENT_006 | 2022-09-01 → 2022-09-17 | 2022-09-20 → 2022-10-02
EVENT_007 | 2023-07-05 → 2023-07-20 | 2023-07-23 → 2023-08-05


In [27]:
# ============================================================
# 26. SENTINEL-1 FLOOD DETECTION FUNCTION
# ============================================================

def sentinel1_collection(
    start_date,
    end_date
):

    return (
        ee.ImageCollection(
            'COPERNICUS/S1_GRD'
        )
        .filterBounds(
            STUDY_ROI
        )
        .filterDate(
            start_date,
            end_date
        )
        .filter(
            ee.Filter.eq(
                'instrumentMode',
                'IW'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VV'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VH'
            )
        )
        .select(
            ['VV', 'VH']
        )
    )


def process_flood_event(event):

    pre = sentinel1_collection(
        event['pre_start'],
        event['pre_end']
    )

    post = sentinel1_collection(
        event['post_start'],
        event['post_end']
    )

    pre_image = (
        pre
        .median()
    )

    post_image = (
        post
        .median()
    )

    VV_pre = (
        pre_image
        .select('VV')
        .rename('VV_pre')
    )

    VV_post = (
        post_image
        .select('VV')
        .rename('VV_post')
    )

    VH_pre = (
        pre_image
        .select('VH')
        .rename('VH_pre')
    )

    VH_post = (
        post_image
        .select('VH')
        .rename('VH_post')
    )


    # --------------------------------------------------------
    # Backscatter changes
    # --------------------------------------------------------

    VV_change = (
        VV_post
        .subtract(VV_pre)
        .rename('VV_change')
    )

    VH_change = (
        VH_post
        .subtract(VH_pre)
        .rename('VH_change')
    )


    # --------------------------------------------------------
    # Flood candidate
    # --------------------------------------------------------

    flood_candidate = (
        VV_change
        .lt(-1.5)
        .And(
            VH_change
            .lt(-1.5)
        )
    )


    # --------------------------------------------------------
    # Exclude permanent water
    # --------------------------------------------------------

    FloodMask = (
        flood_candidate
        .And(
            PermanentWater.Not()
        )
        .rename('Flood')
    )


    # --------------------------------------------------------
    # Remove isolated single-pixel noise
    # --------------------------------------------------------

    FloodMask = (
        FloodMask
        .selfMask()
        .connectedPixelCount(
            maxSize=100,
            eightConnected=True
        )
        .gte(8)
        .rename('Flood')
    )


    return (
        ee.Image.cat(
            [
                VV_pre,
                VV_post,
                VH_pre,
                VH_post,
                VV_change,
                VH_change,
                FloodMask
            ]
        )
        .clip(STUDY_ROI)
        .set(
            {
                'event_id':
                    event['event_id'],

                'pre_start':
                    event['pre_start'],

                'pre_end':
                    event['pre_end'],

                'post_start':
                    event['post_start'],

                'post_end':
                    event['post_end']
            }
        )
    )

In [28]:
# ============================================================
# 27. PROCESS VALID FLOOD EVENTS
# ============================================================

FLOOD_EVENT_IMAGES = []

for event in VALID_EVENTS:

    print(
        "Processing:",
        event['event_id']
    )

    image = (
        process_flood_event(
            event
        )
    )

    FLOOD_EVENT_IMAGES.append(
        image
    )


print(
    "Processed events:",
    len(FLOOD_EVENT_IMAGES)
)

Processing: EVENT_001
Processing: EVENT_002
Processing: EVENT_003
Processing: EVENT_004
Processing: EVENT_005
Processing: EVENT_006
Processing: EVENT_007
Processed events: 7


In [29]:
# ============================================================
# 28. FLOOD EVENT COLLECTION
# ============================================================

FLOOD_COLLECTION = (
    ee.ImageCollection
    .fromImages(
        FLOOD_EVENT_IMAGES
    )
)

print(
    "Flood collection size:",
    FLOOD_COLLECTION
    .size()
    .getInfo()
)

Flood collection size: 7


In [30]:
# ============================================================
# 29. MASTER FLOOD INVENTORY
# ============================================================

FloodFrequency = (
    FLOOD_COLLECTION
    .select('Flood')
    .sum()
    .rename(
        'FloodFrequency'
    )
)


FloodInventory = (
    FloodFrequency
    .gt(0)
    .And(
        PermanentWater.Not()
    )
    .rename(
        'FloodInventory'
    )
    .clip(STUDY_ROI)
)


RecurrentFlood = (
    FloodFrequency
    .gte(2)
    .rename(
        'RecurrentFlood'
    )
    .clip(STUDY_ROI)
)


Map.addLayer(
    FloodFrequency,
    {
        'min': 0,
        'max': max(
            len(VALID_EVENTS),
            1
        ),
        'palette': [
            'ffffff',
            'ffff00',
            'ff9900',
            'ff3300',
            '990000'
        ]
    },
    'Flood Frequency'
)


Map.addLayer(
    FloodInventory,
    {
        'palette': [
            'ff0000'
        ]
    },
    'Master Flood Inventory'
)

In [5]:
# ============================================================
# 1. ENVIRONMENT AND GOOGLE EARTH ENGINE INITIALIZATION
# ============================================================

# Install required packages
!pip -q install geemap earthengine-api xgboost scikit-learn pandas numpy matplotlib seaborn

# ------------------------------------------------------------
# Python imports
# ------------------------------------------------------------

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    brier_score_loss
)

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier

print("Python libraries imported successfully.")


# ------------------------------------------------------------
# Google Earth Engine authentication
# ------------------------------------------------------------

try:
    ee.Initialize(
        project='weighty-tensor-441119-q2',
        opt_url='https://earthengine-highvolume.googleapis.com'
    )

    print("Google Earth Engine was already initialized.")

except Exception:

    print("Earth Engine authentication is required.")

    ee.Authenticate()

    ee.Initialize(
        project='weighty-tensor-441119-q2',
        opt_url='https://earthengine-highvolume.googleapis.com'
    )

    print("Google Earth Engine authenticated and initialized successfully.")


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("Earth Engine API version:", ee.__version__)

print("Initialization test:")

print(
    ee.Image('NASA/NASADEM_HGT/001')
    .select('elevation')
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=ee.Geometry.Point([9.7, 4.0]),
        scale=30,
        maxPixels=1e6
    )
    .getInfo()
)

print("SECTION 1 COMPLETED SUCCESSFULLY.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.8 MB/s eta 0:00:00
Python libraries imported successfully.
Earth Engine authentication is required.
Google Earth Engine authenticated and initialized successfully.
Earth Engine API version: 1.7.35
Initialization test:
{'elevation': 0}
SECTION 1 COMPLETED SUCCESSFULLY.


In [6]:
# ============================================================
# 2. GLOBAL STUDY PARAMETERS
# ============================================================

STUDY_NAME = "Coastal Cameroon Flood Susceptibility"

# ------------------------------------------------------------
# Analysis projection
# ------------------------------------------------------------

ANALYSIS_CRS = 'EPSG:32632'

# ------------------------------------------------------------
# Primary modelling resolution
# ------------------------------------------------------------

ANALYSIS_SCALE = 30

# ------------------------------------------------------------
# Current analysis date
# ------------------------------------------------------------

TODAY = '2026-08-09'

DATA_CUTOFF = '2026-08-09'

# ------------------------------------------------------------
# Sentinel-2 analysis period
# ------------------------------------------------------------

S2_START = '2021-01-01'
S2_END = '2026-08-09'

# ------------------------------------------------------------
# CHIRPS analysis period
# ------------------------------------------------------------

CHIRPS_START = '1981-01-01'
CHIRPS_END = '2026-08-09'

# ------------------------------------------------------------
# GPM analysis period
# ------------------------------------------------------------

GPM_START = '1998-01-01'
GPM_END = '2026-08-09'

# ------------------------------------------------------------
# Sentinel-1 availability period
# ------------------------------------------------------------

S1_START = '2015-01-01'
S1_END = '2026-08-09'

# ------------------------------------------------------------
# Print parameters
# ------------------------------------------------------------

print("============================================================")
print("GLOBAL STUDY PARAMETERS")
print("============================================================")

print("Study name:", STUDY_NAME)
print("Analysis CRS:", ANALYSIS_CRS)
print("Analysis scale:", ANALYSIS_SCALE, "m")
print("Current date:", TODAY)
print("Data cutoff:", DATA_CUTOFF)

print("Sentinel-1:", S1_START, "to", S1_END)
print("Sentinel-2:", S2_START, "to", S2_END)
print("CHIRPS:", CHIRPS_START, "to", CHIRPS_END)
print("GPM:", GPM_START, "to", GPM_END)

print("SECTION 2 COMPLETED SUCCESSFULLY.")

GLOBAL STUDY PARAMETERS
Study name: Coastal Cameroon Flood Susceptibility
Analysis CRS: EPSG:32632
Analysis scale: 30 m
Current date: 2026-08-09
Data cutoff: 2026-08-09
Sentinel-1: 2015-01-01 to 2026-08-09
Sentinel-2: 2021-01-01 to 2026-08-09
CHIRPS: 1981-01-01 to 2026-08-09
GPM: 1998-01-01 to 2026-08-09
SECTION 2 COMPLETED SUCCESSFULLY.


In [7]:
# ============================================================
# 3. DEFINITIVE COASTAL CAMEROON STUDY ROI
# ============================================================

# This is the single authoritative study-region variable.
# All subsequent sections will use STUDY_ROI.
#
# Do NOT rename this variable.
# Do NOT use analysis_region or cameroon_coast later.

STUDY_ROI = ee.Geometry.Polygon(
    [[
        [8.668213, 2.547988],
        [8.668213, 4.861101],
        [10.645752, 4.861101],
        [10.645752, 2.547988],
        [8.668213, 2.547988]
    ]],
    proj='EPSG:4326',
    geodesic=False
)

# ------------------------------------------------------------
# ROI verification
# ------------------------------------------------------------

print("============================================================")
print("STUDY ROI")
print("============================================================")

print("STUDY_ROI successfully created.")

print(
    "Geometry type:",
    STUDY_ROI.type().getInfo()
)

print(
    "ROI coordinates:",
    STUDY_ROI.coordinates().getInfo()
)

print("SECTION 3 COMPLETED SUCCESSFULLY.")

STUDY ROI
STUDY_ROI successfully created.
Geometry type: Polygon
ROI coordinates: [[[8.668213, 2.547988], [8.668213, 4.861101], [10.645752, 4.861101], [10.645752, 2.547988], [8.668213, 2.547988]]]
SECTION 3 COMPLETED SUCCESSFULLY.


In [8]:
# ============================================================
# 4. STUDY ROI MAP AND AREA VERIFICATION
# ============================================================

# This section requires:
#   - ee
#   - geemap
#   - STUDY_ROI
#   - ANALYSIS_SCALE
#
# These were created in Sections 1–3.

# ------------------------------------------------------------
# 4.1 Create interactive map
# ------------------------------------------------------------

Map = geemap.Map(
    basemap='SATELLITE'
)

Map.centerObject(
    STUDY_ROI,
    8
)

# ------------------------------------------------------------
# 4.2 Add study boundary
# ------------------------------------------------------------

Map.addLayer(
    STUDY_ROI,
    {
        'color': 'red'
    },
    'Coastal Cameroon Study ROI'
)

# ------------------------------------------------------------
# 4.3 Calculate ROI area
# ------------------------------------------------------------

ROI_AREA_M2 = (
    STUDY_ROI
    .area(
        maxError=100
    )
)

ROI_AREA_KM2 = (
    ROI_AREA_M2
    .divide(1e6)
)

# ------------------------------------------------------------
# 4.4 Print area
# ------------------------------------------------------------

print("============================================================")
print("STUDY REGION VERIFICATION")
print("============================================================")

print(
    "Study area (km²):",
    ROI_AREA_KM2.getInfo()
)

print(
    "Analysis scale:",
    ANALYSIS_SCALE,
    "m"
)

print(
    "Analysis CRS:",
    ANALYSIS_CRS
)

# ------------------------------------------------------------
# 4.5 Display map
# ------------------------------------------------------------

Map

STUDY REGION VERIFICATION
Study area (km²): 56432.58556156796
Analysis scale: 30 m
Analysis CRS: EPSG:32632


Map(center=[3.704223963390629, 9.656982499999982], controls=(WidgetControl(options=['position', 'transparent_b…

In [9]:
# ============================================================
# 5. DATASET AVAILABILITY AND SPATIAL COVERAGE VALIDATION
# ============================================================

# This section requires:
#   - ee
#   - STUDY_ROI
#   - TODAY
#   - S1_START
#   - S1_END
#   - S2_START
#   - S2_END
#   - CHIRPS_START
#   - CHIRPS_END
#   - GPM_START
#   - GPM_END
#
# All were created in Sections 1–3.

print("============================================================")
print("DATASET AVAILABILITY VALIDATION")
print("============================================================")


# ------------------------------------------------------------
# 5.1 SENTINEL-1 GRD
# ------------------------------------------------------------

S1_TEST = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(STUDY_ROI)
    .filterDate(
        S1_START,
        S1_END
    )
)

S1_COUNT = S1_TEST.size().getInfo()

print(
    "Sentinel-1 GRD images:",
    S1_COUNT
)


# ------------------------------------------------------------
# 5.2 SENTINEL-2 SR HARMONIZED
# ------------------------------------------------------------

S2_TEST = (
    ee.ImageCollection(
        'COPERNICUS/S2_SR_HARMONIZED'
    )
    .filterBounds(STUDY_ROI)
    .filterDate(
        S2_START,
        S2_END
    )
)

S2_COUNT = S2_TEST.size().getInfo()

print(
    "Sentinel-2 SR Harmonized images:",
    S2_COUNT
)


# ------------------------------------------------------------
# 5.3 CHIRPS DAILY PRECIPITATION
# ------------------------------------------------------------

CHIRPS_TEST = (
    ee.ImageCollection(
        'UCSB-CHG/CHIRPS/DAILY'
    )
    .filterBounds(STUDY_ROI)
    .filterDate(
        CHIRPS_START,
        CHIRPS_END
    )
)

CHIRPS_COUNT = CHIRPS_TEST.size().getInfo()

print(
    "CHIRPS daily images:",
    CHIRPS_COUNT
)


# ------------------------------------------------------------
# 5.4 GPM IMERG V07
# ------------------------------------------------------------

GPM_TEST = (
    ee.ImageCollection(
        'NASA/GPM_L3/IMERG_V07'
    )
    .filterBounds(STUDY_ROI)
    .filterDate(
        GPM_START,
        GPM_END
    )
)

GPM_COUNT = GPM_TEST.size().getInfo()

print(
    "GPM IMERG V07 images:",
    GPM_COUNT
)


# ------------------------------------------------------------
# 5.5 DYNAMIC WORLD
# ------------------------------------------------------------

DW_TEST = (
    ee.ImageCollection(
        'GOOGLE/DYNAMICWORLD/V1'
    )
    .filterBounds(STUDY_ROI)
    .filterDate(
        '2021-01-01',
        TODAY
    )
)

DW_COUNT = DW_TEST.size().getInfo()

print(
    "Dynamic World images:",
    DW_COUNT
)


# ------------------------------------------------------------
# 5.6 ERA5-LAND HOURLY
# ------------------------------------------------------------

ERA5_TEST = (
    ee.ImageCollection(
        'ECMWF/ERA5_LAND/HOURLY'
    )
    .filterBounds(STUDY_ROI)
    .filterDate(
        '2015-01-01',
        TODAY
    )
)

ERA5_COUNT = ERA5_TEST.size().getInfo()

print(
    "ERA5-Land hourly images:",
    ERA5_COUNT
)


# ------------------------------------------------------------
# 5.7 JRC GLOBAL SURFACE WATER
# ------------------------------------------------------------

GSW_TEST = ee.Image(
    'JRC/GSW1_4/GlobalSurfaceWater'
)

GSW_BANDS = (
    GSW_TEST
    .bandNames()
    .getInfo()
)

print(
    "JRC Global Surface Water bands:",
    GSW_BANDS
)


# ------------------------------------------------------------
# 5.8 NASADEM
# ------------------------------------------------------------

NASADEM_TEST = ee.Image(
    'NASA/NASADEM_HGT/001'
)

NASADEM_BANDS = (
    NASADEM_TEST
    .bandNames()
    .getInfo()
)

print(
    "NASADEM bands:",
    NASADEM_BANDS
)


# ------------------------------------------------------------
# 5.9 SUMMARY TABLE
# ------------------------------------------------------------

availability = pd.DataFrame({

    'Dataset': [
        'Sentinel-1 GRD',
        'Sentinel-2 SR Harmonized',
        'CHIRPS Daily',
        'GPM IMERG V07',
        'Dynamic World',
        'ERA5-Land Hourly',
        'JRC Global Surface Water',
        'NASADEM'
    ],

    'Type': [
        'ImageCollection',
        'ImageCollection',
        'ImageCollection',
        'ImageCollection',
        'ImageCollection',
        'ImageCollection',
        'Image',
        'Image'
    ],

    'Images_or_Bands': [
        S1_COUNT,
        S2_COUNT,
        CHIRPS_COUNT,
        GPM_COUNT,
        DW_COUNT,
        ERA5_COUNT,
        len(GSW_BANDS),
        len(NASADEM_BANDS)
    ]
})

display(availability)


# ------------------------------------------------------------
# 5.10 HARD VALIDATION
# ------------------------------------------------------------

if S1_COUNT == 0:
    raise RuntimeError(
        "Sentinel-1 contains no images over STUDY_ROI."
    )

if S2_COUNT == 0:
    raise RuntimeError(
        "Sentinel-2 contains no images over STUDY_ROI."
    )

if CHIRPS_COUNT == 0:
    raise RuntimeError(
        "CHIRPS contains no images over STUDY_ROI."
    )

if GPM_COUNT == 0:
    raise RuntimeError(
        "GPM IMERG contains no images over STUDY_ROI."
    )

if DW_COUNT == 0:
    raise RuntimeError(
        "Dynamic World contains no images over STUDY_ROI."
    )

if ERA5_COUNT == 0:
    raise RuntimeError(
        "ERA5-Land contains no images over STUDY_ROI."
    )

if 'elevation' not in NASADEM_BANDS:
    raise RuntimeError(
        "NASADEM elevation band was not found."
    )

if 'occurrence' not in GSW_BANDS:
    raise RuntimeError(
        "JRC Global Surface Water occurrence band was not found."
    )


print()
print("============================================================")
print("SECTION 5 VALIDATION PASSED")
print("============================================================")
print("All core datasets are available over STUDY_ROI.")
print("No predictor has been constructed yet.")
print("Ready for Section 6.")

DATASET AVAILABILITY VALIDATION
Sentinel-1 GRD images: 3458
Sentinel-2 SR Harmonized images: 7942
CHIRPS daily images: 16617
GPM IMERG V07 images: 501414
Dynamic World images: 1226
ERA5-Land hourly images: 101589
JRC Global Surface Water bands: ['occurrence', 'change_abs', 'change_norm', 'seasonality', 'recurrence', 'transition', 'max_extent']
NASADEM bands: ['elevation', 'num', 'swb']


,Dataset,Type,Images_or_Bands
0,Sentinel-1 GRD,ImageCollection,3458
1,Sentinel-2 SR Harmonized,ImageCollection,7942
2,CHIRPS Daily,ImageCollection,16617
3,GPM IMERG V07,ImageCollection,501414
4,Dynamic World,ImageCollection,1226
5,ERA5-Land Hourly,ImageCollection,101589
6,JRC Global Surface Water,Image,7
7,NASADEM,Image,3



SECTION 5 VALIDATION PASSED
All core datasets are available over STUDY_ROI.
No predictor has been constructed yet.
Ready for Section 6.


In [10]:
# ============================================================
# 6. NASADEM TERRAIN PREDICTORS
# ============================================================

# Requires:
#   - ee
#   - STUDY_ROI
#   - ANALYSIS_SCALE
#   - ANALYSIS_CRS
#
# Created in Sections 1–5.

print("============================================================")
print("SECTION 6 — NASADEM TERRAIN PREDICTORS")
print("============================================================")


# ------------------------------------------------------------
# 6.1 LOAD NASADEM
# ------------------------------------------------------------

NASADEM = (
    ee.Image('NASA/NASADEM_HGT/001')
    .select('elevation')
    .clip(STUDY_ROI)
)

print("NASADEM loaded.")


# ------------------------------------------------------------
# 6.2 ELEVATION
# ------------------------------------------------------------

Elevation = (
    NASADEM
    .rename('Elevation')
)

print("Elevation created.")


# ------------------------------------------------------------
# 6.3 SLOPE
# ------------------------------------------------------------

Slope = (
    ee.Terrain
    .slope(Elevation)
    .rename('Slope')
)

print("Slope created.")


# ------------------------------------------------------------
# 6.4 ASPECT
# ------------------------------------------------------------

Aspect = (
    ee.Terrain
    .aspect(Elevation)
    .rename('Aspect')
)

print("Aspect created.")


# ------------------------------------------------------------
# 6.5 TERRAIN RUGGEDNESS INDEX — TRI
# ------------------------------------------------------------

TRI_KERNEL = ee.Kernel.square(
    radius=1,
    units='pixels',
    normalize=False
)

Elevation_Mean_3x3 = (
    Elevation
    .reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=TRI_KERNEL
    )
)

TRI = (
    Elevation
    .subtract(Elevation_Mean_3x3)
    .abs()
    .rename('TRI')
)

print("TRI created.")


# ------------------------------------------------------------
# 6.6 CURVATURE
# ------------------------------------------------------------

CURVATURE_KERNEL = ee.Kernel.laplacian8(
    normalize=False
)

Curvature = (
    Elevation
    .convolve(CURVATURE_KERNEL)
    .rename('Curvature')
)

print("Curvature created.")


# ------------------------------------------------------------
# 6.7 TOPOGRAPHIC POSITION INDEX — TPI
# ------------------------------------------------------------

TPI_KERNEL = ee.Kernel.circle(
    radius=300,
    units='meters',
    normalize=False
)

Elevation_Mean_300m = (
    Elevation
    .reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=TPI_KERNEL
    )
)

TPI = (
    Elevation
    .subtract(Elevation_Mean_300m)
    .rename('TPI')
)

print("TPI created.")


# ------------------------------------------------------------
# 6.8 CREATE TERRAIN STACK
# ------------------------------------------------------------

TERRAIN_STACK = ee.Image.cat([
    Elevation,
    Slope,
    Aspect,
    TRI,
    Curvature,
    TPI
]).clip(STUDY_ROI)


# ------------------------------------------------------------
# 6.9 VERIFY BAND NAMES
# ------------------------------------------------------------

TERRAIN_BANDS = (
    TERRAIN_STACK
    .bandNames()
    .getInfo()
)

print()
print("Terrain predictor bands:")
print(TERRAIN_BANDS)


# ------------------------------------------------------------
# 6.10 VERIFY EXPECTED BANDS
# ------------------------------------------------------------

EXPECTED_TERRAIN_BANDS = [
    'Elevation',
    'Slope',
    'Aspect',
    'TRI',
    'Curvature',
    'TPI'
]

missing_terrain = [
    band
    for band in EXPECTED_TERRAIN_BANDS
    if band not in TERRAIN_BANDS
]

if len(missing_terrain) > 0:

    raise RuntimeError(
        "Terrain predictor creation failed. "
        f"Missing bands: {missing_terrain}"
    )


# ------------------------------------------------------------
# 6.11 BASIC NUMERICAL VALIDATION
# ------------------------------------------------------------

TERRAIN_STATS = (
    TERRAIN_STACK
    .reduceRegion(
        reducer=ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.minMax(),
            sharedInputs=True
        ),
        geometry=STUDY_ROI,
        scale=ANALYSIS_SCALE,
        maxPixels=1e8,
        bestEffort=True
    )
)

TERRAIN_STATS_INFO = TERRAIN_STATS.getInfo()

print()
print("Terrain statistics:")
print(TERRAIN_STATS_INFO)


# ------------------------------------------------------------
# 6.12 MAP VISUALIZATION
# ------------------------------------------------------------

Map.addLayer(
    Elevation,
    {
        'min': 0,
        'max': 1000,
        'palette': [
            '006400',
            '7FFF00',
            'FFFF00',
            'FFA500',
            'A52A2A',
            'FFFFFF'
        ]
    },
    'Elevation'
)

Map.addLayer(
    Slope,
    {
        'min': 0,
        'max': 30,
        'palette': [
            '006400',
            'ADFF2F',
            'FFFF00',
            'FFA500',
            'FF0000'
        ]
    },
    'Slope'
)

Map.addLayer(
    TPI,
    {
        'min': -50,
        'max': 50,
        'palette': [
            '0000FF',
            'FFFFFF',
            'FF0000'
        ]
    },
    'TPI'
)


# ------------------------------------------------------------
# 6.13 FINAL CHECK
# ------------------------------------------------------------

print()
print("============================================================")
print("SECTION 6 COMPLETED SUCCESSFULLY")
print("============================================================")
print("Created variables:")
print("  Elevation")
print("  Slope")
print("  Aspect")
print("  TRI")
print("  Curvature")
print("  TPI")
print("  TERRAIN_STACK")
print()
print("Terrain bands verified successfully.")

SECTION 6 — NASADEM TERRAIN PREDICTORS
NASADEM loaded.
Elevation created.
Slope created.
Aspect created.
TRI created.
Curvature created.
TPI created.

Terrain predictor bands:
['Elevation', 'Slope', 'Aspect', 'TRI', 'Curvature', 'TPI']

Terrain statistics:
{'Aspect_max': 359.626953125, 'Aspect_mean': 116.4758077342956, 'Aspect_min': 0, 'Curvature_max': 651, 'Curvature_mean': -0.30610351671574276, 'Curvature_min': -4653, 'Elevation_max': 4023, 'Elevation_mean': 160.0883615336551, 'Elevation_min': -48, 'Slope_max': 69.28428649902344, 'Slope_mean': 4.499209872292185, 'Slope_min': 0, 'TPI_max': 187.68539325842698, 'TPI_mean': 0.0002866200186359592, 'TPI_min': -164.70786516853923, 'TRI_max': 75.77777777777783, 'TRI_mean': 1.441249766571092, 'TRI_min': 0}

SECTION 6 COMPLETED SUCCESSFULLY
Created variables:
  Elevation
  Slope
  Aspect
  TRI
  Curvature
  TPI
  TERRAIN_STACK

Terrain bands verified successfully.


In [11]:
# ============================================================
# 7. MERIT HYDROGRAPHY AND HYDROLOGICAL PREDICTORS
# ============================================================

# Requires:
#   - ee
#   - STUDY_ROI
#   - ANALYSIS_SCALE
#   - Elevation
#   - Slope
#
# Created in Sections 1–6.

print("============================================================")
print("SECTION 7 — MERIT HYDROGRAPHY")
print("============================================================")


# ------------------------------------------------------------
# 7.1 LOAD MERIT HYDRO
# ------------------------------------------------------------

MERIT = (
    ee.Image('MERIT/Hydro/v1_0_1')
    .clip(STUDY_ROI)
)

print("MERIT Hydro loaded successfully.")


# ------------------------------------------------------------
# 7.2 VERIFY ACTUAL MERIT BANDS
# ------------------------------------------------------------

MERIT_BANDS = (
    MERIT
    .bandNames()
    .getInfo()
)

print()
print("Available MERIT Hydro bands:")
print(MERIT_BANDS)


# ------------------------------------------------------------
# 7.3 VERIFY REQUIRED BANDS
# ------------------------------------------------------------

REQUIRED_MERIT_BANDS = [
    'upa',
    'hnd',
    'wth'
]

MISSING_MERIT_BANDS = [
    band
    for band in REQUIRED_MERIT_BANDS
    if band not in MERIT_BANDS
]

if len(MISSING_MERIT_BANDS) > 0:

    raise RuntimeError(
        "Required MERIT Hydro bands are missing: "
        + str(MISSING_MERIT_BANDS)
    )

print()
print("Required MERIT bands verified.")


# ------------------------------------------------------------
# 7.4 UPSTREAM DRAINAGE AREA / FLOW ACCUMULATION
# ------------------------------------------------------------

FlowAccumulation = (
    MERIT
    .select('upa')
    .rename('FlowAccumulation')
)

print("FlowAccumulation created.")


# ------------------------------------------------------------
# 7.5 LOG-TRANSFORMED FLOW ACCUMULATION
# ------------------------------------------------------------

FlowAccumulation_Log = (
    FlowAccumulation
    .add(1)
    .log10()
    .rename('FlowAccumulation_Log')
)

print("FlowAccumulation_Log created.")


# ------------------------------------------------------------
# 7.6 HEIGHT ABOVE NEAREST DRAINAGE — HAND
# ------------------------------------------------------------

HAND = (
    MERIT
    .select('hnd')
    .rename('HAND')
)

print("HAND created.")


# ------------------------------------------------------------
# 7.7 RIVER WIDTH
# ------------------------------------------------------------

RiverWidth = (
    MERIT
    .select('wth')
    .rename('RiverWidth')
)

print("RiverWidth created.")


# ------------------------------------------------------------
# 7.8 CREATE INITIAL HYDROLOGICAL STACK
# ------------------------------------------------------------

HYDRO_STACK = (
    ee.Image.cat([
        FlowAccumulation,
        FlowAccumulation_Log,
        HAND,
        RiverWidth
    ])
    .clip(STUDY_ROI)
)


# ------------------------------------------------------------
# 7.9 VERIFY HYDROLOGICAL BANDS
# ------------------------------------------------------------

HYDRO_BANDS = (
    HYDRO_STACK
    .bandNames()
    .getInfo()
)

print()
print("Hydrological predictor bands:")
print(HYDRO_BANDS)


EXPECTED_HYDRO_BANDS = [
    'FlowAccumulation',
    'FlowAccumulation_Log',
    'HAND',
    'RiverWidth'
]

MISSING_HYDRO_BANDS = [
    band
    for band in EXPECTED_HYDRO_BANDS
    if band not in HYDRO_BANDS
]

if len(MISSING_HYDRO_BANDS) > 0:

    raise RuntimeError(
        "Hydrological stack is incomplete. "
        f"Missing: {MISSING_HYDRO_BANDS}"
    )


# ------------------------------------------------------------
# 7.10 BASIC STATISTICAL VALIDATION
# ------------------------------------------------------------

HYDRO_STATS = (
    HYDRO_STACK
    .reduceRegion(
        reducer=ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.minMax(),
            sharedInputs=True
        ),
        geometry=STUDY_ROI,
        scale=ANALYSIS_SCALE,
        maxPixels=1e8,
        bestEffort=True
    )
)

HYDRO_STATS_INFO = HYDRO_STATS.getInfo()

print()
print("Hydrological statistics:")
print(HYDRO_STATS_INFO)


# ------------------------------------------------------------
# 7.11 VISUALIZE FLOW ACCUMULATION
# ------------------------------------------------------------

Map.addLayer(
    FlowAccumulation_Log,
    {
        'min': 0,
        'max': 6,
        'palette': [
            'FFFFFF',
            'D9F0FF',
            '74ADD1',
            '4575B4',
            '313695',
            '000000'
        ]
    },
    'Log Flow Accumulation'
)


# ------------------------------------------------------------
# 7.12 VISUALIZE HAND
# ------------------------------------------------------------

Map.addLayer(
    HAND,
    {
        'min': 0,
        'max': 50,
        'palette': [
            '0000FF',
            '00FFFF',
            'FFFF00',
            'FF0000'
        ]
    },
    'HAND'
)


# ------------------------------------------------------------
# 7.13 FINAL VALIDATION
# ------------------------------------------------------------

print()
print("============================================================")
print("SECTION 7 COMPLETED SUCCESSFULLY")
print("============================================================")

print("Created variables:")
print("  MERIT")
print("  FlowAccumulation")
print("  FlowAccumulation_Log")
print("  HAND")
print("  RiverWidth")
print("  HYDRO_STACK")

print()
print("MERIT bands verified before predictor construction.")

SECTION 7 — MERIT HYDROGRAPHY
MERIT Hydro loaded successfully.

Available MERIT Hydro bands:
['elv', 'dir', 'wth', 'wat', 'upa', 'upg', 'hnd', 'viswth']

Required MERIT bands verified.
FlowAccumulation created.
FlowAccumulation_Log created.
HAND created.
RiverWidth created.

Hydrological predictor bands:
['FlowAccumulation', 'FlowAccumulation_Log', 'HAND', 'RiverWidth']

Hydrological statistics:
{'FlowAccumulation_Log_max': 5.122971446147787, 'FlowAccumulation_Log_mean': 0.08634000199534547, 'FlowAccumulation_Log_min': 0.0036835360413548968, 'FlowAccumulation_max': 132729.71875, 'FlowAccumulation_mean': 68.03890010425073, 'FlowAccumulation_min': 0.008517726324498653, 'HAND_max': 2763.60009765625, 'HAND_mean': 41.964060176563066, 'HAND_min': 0, 'RiverWidth_max': 2399.63232421875, 'RiverWidth_mean': 182.29966918376988, 'RiverWidth_min': 0.8999999761581421}

SECTION 7 COMPLETED SUCCESSFULLY
Created variables:
  MERIT
  FlowAccumulation
  FlowAccumulation_Log
  HAND
  RiverWidth
  HYDRO_ST

In [21]:
# ============================================================
# SECTION 30A
# CORE TERRAIN PREDICTORS
# ============================================================
#
# Creates and validates:
# Elevation
# Slope
# Aspect
# Curvature
# TPI
# TRI
#
# This section is intentionally independent of the previous
# Section 30 script.
# ============================================================

print("=" * 70)
print("SECTION 30A — CORE TERRAIN PREDICTORS")
print("=" * 70)


# ------------------------------------------------------------
# 30A.1 — Earth Engine check
# ------------------------------------------------------------

try:
    import ee
except ImportError:
    raise ImportError(
        "Earth Engine API is not installed."
    )


try:
    ee.Number(1).getInfo()
    print("Earth Engine connection: OK")

except Exception:

    ee.Initialize(
        project='weighty-tensor-441119-q2',
        opt_url='https://earthengine-highvolume.googleapis.com'
    )

    print(
        "Earth Engine initialized successfully."
    )


# ------------------------------------------------------------
# 30A.2 — Analysis parameters
# ------------------------------------------------------------

ANALYSIS_SCALE = 30
ANALYSIS_CRS = 'EPSG:32632'


# ------------------------------------------------------------
# 30A.3 — Study ROI
# ------------------------------------------------------------

STUDY_ROI = ee.Geometry.Polygon(
    [[
        [8.668213, 2.547988],
        [8.668213, 4.861101],
        [10.645752, 4.861101],
        [10.645752, 2.547988],
        [8.668213, 2.547988]
    ]],
    proj='EPSG:4326',
    geodesic=False
)


print(
    "STUDY_ROI: OK"
)


# ============================================================
# 30A.4 — NASADEM
# ============================================================

NASADEM_30A = (
    ee.Image(
        'NASA/NASADEM_HGT/001'
    )
    .select('elevation')
    .clip(STUDY_ROI)
)


Elevation = (
    NASADEM_30A
    .rename('Elevation')
)


print(
    "Elevation: OK"
)


# ============================================================
# 30A.5 — Slope
# ============================================================

Slope = (
    ee.Terrain
    .slope(Elevation)
    .rename('Slope')
)


print(
    "Slope: OK"
)


# ============================================================
# 30A.6 — Aspect
# ============================================================

Aspect = (
    ee.Terrain
    .aspect(Elevation)
    .rename('Aspect')
)


print(
    "Aspect: OK"
)


# ============================================================
# 30A.7 — Curvature
# ============================================================

Curvature = (
    Elevation
    .convolve(
        ee.Kernel.laplacian8(
            normalize=False
        )
    )
    .rename('Curvature')
)


print(
    "Curvature: OK"
)


# ============================================================
# 30A.8 — TPI
# ============================================================

TPI_KERNEL = ee.Kernel.circle(
    radius=300,
    units='meters',
    normalize=False
)


ElevationMean = (
    Elevation
    .reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=TPI_KERNEL
    )
)


TPI = (
    Elevation
    .subtract(
        ElevationMean
    )
    .rename('TPI')
)


print(
    "TPI: OK"
)


# ============================================================
# 30A.9 — TRI
# ============================================================

TRI_KERNEL = ee.Kernel.square(
    radius=1,
    units='pixels',
    normalize=False
)


ElevationMeanTRI = (
    Elevation
    .reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=TRI_KERNEL
    )
)


TRI = (
    Elevation
    .subtract(
        ElevationMeanTRI
    )
    .abs()
    .rename('TRI')
)


print(
    "TRI: OK"
)


# ============================================================
# 30A.10 — Create terrain stack
# ============================================================

TERRAIN_STACK_30A = ee.Image.cat([
    Elevation,
    Slope,
    Aspect,
    Curvature,
    TPI,
    TRI
]).clip(STUDY_ROI)


# ============================================================
# 30A.11 — Validate
# ============================================================

TERRAIN_BANDS_30A = (
    TERRAIN_STACK_30A
    .bandNames()
    .getInfo()
)


EXPECTED_TERRAIN_BANDS_30A = [
    'Elevation',
    'Slope',
    'Aspect',
    'Curvature',
    'TPI',
    'TRI'
]


missing = [
    b
    for b in EXPECTED_TERRAIN_BANDS_30A
    if b not in TERRAIN_BANDS_30A
]


if missing:

    raise RuntimeError(
        "SECTION 30A FAILED.\n"
        "Missing terrain bands:\n"
        + "\n".join(missing)
    )


# ============================================================
# 30A.12 — Final status
# ============================================================

print()
print("Terrain bands successfully created:")
print()

for band in TERRAIN_BANDS_30A:
    print("  ✓", band)

print()
print("=" * 70)
print("SECTION 30A COMPLETED SUCCESSFULLY")
print("=" * 70)

SECTION 30A — CORE TERRAIN PREDICTORS
Earth Engine connection: OK
STUDY_ROI: OK
Elevation: OK
Slope: OK
Aspect: OK
Curvature: OK
TPI: OK
TRI: OK

Terrain bands successfully created:

  ✓ Elevation
  ✓ Slope
  ✓ Aspect
  ✓ Curvature
  ✓ TPI
  ✓ TRI

SECTION 30A COMPLETED SUCCESSFULLY


In [22]:
# ============================================================
# SECTION 30B
# HYDROLOGICAL PREDICTORS — MERIT HYDRO
# ============================================================
#
# PURPOSE
# -------
# Construct and validate the core hydrological predictors
# required for the coastal Cameroon flood-susceptibility model.
#
# VARIABLES CREATED
# -----------------
# 1. FlowAccumulation
# 2. FlowAccumulation_Log
# 3. HAND
# 4. RiverWidth
#
# DEPENDENCIES
# ------------
# Section 30A must have been completed successfully.
#
# REQUIRED EXISTING OBJECTS
# -------------------------
# STUDY_ROI
#
# ============================================================

print("=" * 70)
print("SECTION 30B — HYDROLOGICAL PREDICTORS")
print("=" * 70)


# ============================================================
# 30B.1 — EARTH ENGINE CHECK
# ============================================================

try:

    import ee

    ee.Number(1).getInfo()

    print(
        "Earth Engine connection: OK"
    )

except Exception as e:

    raise RuntimeError(
        "Earth Engine is not available. "
        "Section 30B cannot continue.\n\n"
        f"Error: {e}"
    )


# ============================================================
# 30B.2 — CHECK STUDY ROI
# ============================================================

if 'STUDY_ROI' not in globals():

    raise NameError(
        "STUDY_ROI is not defined. "
        "Section 30B requires the ROI created in Section 30A."
    )


print(
    "STUDY_ROI: OK"
)


# ============================================================
# 30B.3 — LOAD MERIT HYDRO
# ============================================================
#
# MERIT/Hydro/v1_0_1 is an IMAGE.
#
# Therefore we use ee.Image(), not ImageCollection().
#
# ============================================================

print(
    "\nLoading MERIT Hydro..."
)


MERIT_HYDRO_30B = (
    ee.Image(
        'MERIT/Hydro/v1_0_1'
    )
    .clip(
        STUDY_ROI
    )
)


print(
    "MERIT Hydro loaded successfully."
)


# ============================================================
# 30B.4 — FLOW ACCUMULATION
# ============================================================
#
# 'upa' = upstream drainage area.
#
# It represents the accumulated upstream contributing area
# and is used as a hydrological indicator of flow concentration.
#
# ============================================================

FlowAccumulation = (
    MERIT_HYDRO_30B
    .select(
        'upa'
    )
    .rename(
        'FlowAccumulation'
    )
)


print(
    "FlowAccumulation: OK"
)


# ============================================================
# 30B.5 — LOG-TRANSFORMED FLOW ACCUMULATION
# ============================================================
#
# Flow accumulation is highly right-skewed.
#
# log10 transformation compresses extremely large values and
# produces a more stable predictor for statistical / ML analysis.
#
# ============================================================

FlowAccumulation_Log = (
    FlowAccumulation
    .add(
        0.001
    )
    .log10()
    .rename(
        'FlowAccumulation_Log'
    )
)


print(
    "FlowAccumulation_Log: OK"
)


# ============================================================
# 30B.6 — HAND
# ============================================================
#
# HAND = Height Above Nearest Drainage.
#
# Lower HAND generally represents locations closer to drainage
# level and potentially more flood-prone terrain.
#
# ============================================================

HAND = (
    MERIT_HYDRO_30B
    .select(
        'hnd'
    )
    .rename(
        'HAND'
    )
)


print(
    "HAND: OK"
)


# ============================================================
# 30B.7 — RIVER WIDTH
# ============================================================
#
# MERIT Hydro provides river width information.
#
# This is retained as a hydrological predictor and can later
# be evaluated through feature importance / SHAP.
#
# ============================================================

RiverWidth = (
    MERIT_HYDRO_30B
    .select(
        'wth'
    )
    .rename(
        'RiverWidth'
    )
)


print(
    "RiverWidth: OK"
)


# ============================================================
# 30B.8 — CREATE HYDROLOGICAL STACK
# ============================================================

HYDRO_STACK_30B = (
    ee.Image.cat([
        FlowAccumulation,
        FlowAccumulation_Log,
        HAND,
        RiverWidth
    ])
    .clip(
        STUDY_ROI
    )
)


# ============================================================
# 30B.9 — VALIDATE BAND NAMES
# ============================================================

HYDRO_BANDS_30B = (
    HYDRO_STACK_30B
    .bandNames()
    .getInfo()
)


EXPECTED_HYDRO_BANDS_30B = [

    'FlowAccumulation',

    'FlowAccumulation_Log',

    'HAND',

    'RiverWidth'
]


MISSING_HYDRO_BANDS_30B = [

    band

    for band in EXPECTED_HYDRO_BANDS_30B

    if band not in HYDRO_BANDS_30B
]


if len(MISSING_HYDRO_BANDS_30B) > 0:

    raise RuntimeError(
        "SECTION 30B FAILED.\n"
        "The following hydrological bands are missing:\n"
        +
        "\n".join(
            MISSING_HYDRO_BANDS_30B
        )
    )


# ============================================================
# 30B.10 — SERVER-SIDE VALIDATION
# ============================================================
#
# Check that the resulting image actually contains valid
# numerical pixels within the study region.
#
# We use a small diagnostic region rather than exporting data.
#
# ============================================================

HYDRO_VALIDATION = (
    HYDRO_STACK_30B
    .reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=STUDY_ROI,
        scale=30,
        maxPixels=1e8,
        bestEffort=True,
        tileScale=4
    )
    .getInfo()
)


print(
    "\nHydrological pixel-count validation:"
)

for band in HYDRO_BANDS_30B:

    print(
        f"  {band}:",
        HYDRO_VALIDATION.get(
            band,
            0
        )
    )


# ============================================================
# 30B.11 — FINAL STATUS
# ============================================================

print()
print(
    "Hydrological predictor bands successfully created:"
)

for band in HYDRO_BANDS_30B:

    print(
        "  ✓",
        band
    )


print()
print("=" * 70)
print("SECTION 30B COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "Hydrological stack object:",
    "HYDRO_STACK_30B"
)

SECTION 30B — HYDROLOGICAL PREDICTORS
Earth Engine connection: OK
STUDY_ROI: OK

Loading MERIT Hydro...
MERIT Hydro loaded successfully.
FlowAccumulation: OK
FlowAccumulation_Log: OK
HAND: OK
RiverWidth: OK

Hydrological pixel-count validation:
  FlowAccumulation: 15754337
  FlowAccumulation_Log: 15754337
  HAND: 15754337
  RiverWidth: 76664

Hydrological predictor bands successfully created:
  ✓ FlowAccumulation
  ✓ FlowAccumulation_Log
  ✓ HAND
  ✓ RiverWidth

SECTION 30B COMPLETED SUCCESSFULLY
Hydrological stack object: HYDRO_STACK_30B


In [26]:
# ============================================================
# SECTION 31A — EVENT CATALOGUE + SENTINEL-1 VALIDATION
# ============================================================
#
# PURPOSE:
#   1. Make sure Earth Engine and STUDY_ROI exist.
#   2. Recreate the definitive flood-event catalogue.
#   3. Test Sentinel-1 availability for every event.
#   4. Create VALID_EVENTS.
#
# OUTPUT:
#   EVENTS
#   EVENT_STATUS_DF
#   VALID_EVENTS
# ============================================================

print("============================================================")
print("SECTION 31A — EVENT CATALOGUE + S1 VALIDATION")
print("============================================================")


# ------------------------------------------------------------
# 31A.1 — Basic environment check
# ------------------------------------------------------------

if 'ee' not in globals():
    raise NameError(
        "Earth Engine (ee) is not available. "
        "Run the Earth Engine initialization section first."
    )

print("✓ Earth Engine available")


# ------------------------------------------------------------
# 31A.2 — Recreate STUDY_ROI if necessary
# ------------------------------------------------------------

if 'STUDY_ROI' not in globals():

    STUDY_ROI = ee.Geometry.Polygon(
        [[
            [8.668213, 2.547988],
            [8.668213, 4.861101],
            [10.645752, 4.861101],
            [10.645752, 2.547988],
            [8.668213, 2.547988]
        ]],
        proj='EPSG:4326',
        geodesic=False
    )

    print("✓ STUDY_ROI reconstructed")

else:

    print("✓ STUDY_ROI detected")


# ============================================================
# 31A.3 — DEFINITIVE EVENT CATALOGUE
# ============================================================

EVENTS = [

    {
        'event_id': 'EVENT_001',
        'pre_start': '2015-06-01',
        'pre_end': '2015-06-18',
        'post_start': '2015-06-22',
        'post_end': '2015-07-02'
    },

    {
        'event_id': 'EVENT_002',
        'pre_start': '2015-06-20',
        'pre_end': '2015-07-02',
        'post_start': '2015-07-04',
        'post_end': '2015-07-15'
    },

    {
        'event_id': 'EVENT_003',
        'pre_start': '2016-06-01',
        'pre_end': '2016-06-20',
        'post_start': '2016-06-23',
        'post_end': '2016-07-05'
    },

    {
        'event_id': 'EVENT_004',
        'pre_start': '2017-08-25',
        'pre_end': '2017-09-05',
        'post_start': '2017-09-08',
        'post_end': '2017-09-20'
    },

    {
        'event_id': 'EVENT_005',
        'pre_start': '2018-07-10',
        'pre_end': '2018-07-23',
        'post_start': '2018-07-26',
        'post_end': '2018-08-05'
    },

    {
        'event_id': 'EVENT_006',
        'pre_start': '2022-09-01',
        'pre_end': '2022-09-17',
        'post_start': '2022-09-20',
        'post_end': '2022-10-02'
    },

    {
        'event_id': 'EVENT_007',
        'pre_start': '2023-07-05',
        'pre_end': '2023-07-20',
        'post_start': '2023-07-23',
        'post_end': '2023-08-05'
    },

    {
        'event_id': 'EVENT_008_AUG2026',
        'pre_start': '2026-07-20',
        'pre_end': '2026-08-03',
        'post_start': '2026-08-04',
        'post_end': '2026-08-10'
    }
]


print(
    "✓ Event catalogue created:",
    len(EVENTS),
    "events"
)


# ============================================================
# 31A.4 — SENTINEL-1 VALIDATION FUNCTION
# ============================================================

def validate_event_31A(event):

    pre_collection = (
        ee.ImageCollection(
            'COPERNICUS/S1_GRD'
        )
        .filterBounds(
            STUDY_ROI
        )
        .filterDate(
            event['pre_start'],
            event['pre_end']
        )
        .filter(
            ee.Filter.eq(
                'instrumentMode',
                'IW'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VV'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VH'
            )
        )
    )

    post_collection = (
        ee.ImageCollection(
            'COPERNICUS/S1_GRD'
        )
        .filterBounds(
            STUDY_ROI
        )
        .filterDate(
            event['post_start'],
            event['post_end']
        )
        .filter(
            ee.Filter.eq(
                'instrumentMode',
                'IW'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VV'
            )
        )
        .filter(
            ee.Filter.listContains(
                'transmitterReceiverPolarisation',
                'VH'
            )
        )
    )

    return {
        'event_id':
            event['event_id'],

        'pre_start':
            event['pre_start'],

        'pre_end':
            event['pre_end'],

        'post_start':
            event['post_start'],

        'post_end':
            event['post_end'],

        'pre_images':
            pre_collection.size().getInfo(),

        'post_images':
            post_collection.size().getInfo()
    }


# ============================================================
# 31A.5 — VALIDATE ALL EVENTS
# ============================================================

EVENT_STATUS = []

for event in EVENTS:

    print(
        "Checking:",
        event['event_id']
    )

    status = validate_event_31A(event)

    EVENT_STATUS.append(status)


EVENT_STATUS_DF = pd.DataFrame(
    EVENT_STATUS
)


# ============================================================
# 31A.6 — DISPLAY VALIDATION TABLE
# ============================================================

display(
    EVENT_STATUS_DF
)


# ============================================================
# 31A.7 — CREATE VALID_EVENTS
# ============================================================

TODAY_31A = '2026-08-09'

VALID_EVENTS = []

for event, status in zip(
    EVENTS,
    EVENT_STATUS
):

    if (
        status['pre_images'] > 0
        and
        status['post_images'] > 0
        and
        event['post_end'] <= TODAY_31A
    ):

        VALID_EVENTS.append(
            event
        )


# ============================================================
# 31A.8 — FINAL REPORT
# ============================================================

print("\n============================================================")
print("SECTION 31A RESULT")
print("============================================================")

print(
    "Catalogue events:",
    len(EVENTS)
)

print(
    "Valid completed events:",
    len(VALID_EVENTS)
)

for event in VALID_EVENTS:

    print(
        "✓",
        event['event_id'],
        "|",
        event['pre_start'],
        "→",
        event['pre_end'],
        "|",
        event['post_start'],
        "→",
        event['post_end']
    )


# ============================================================
# 31A.9 — HARD VALIDATION
# ============================================================

if len(VALID_EVENTS) == 0:

    raise RuntimeError(
        "SECTION 31A completed but found ZERO valid "
        "completed Sentinel-1 flood events."
    )


print("\n============================================================")
print("SECTION 31A COMPLETED SUCCESSFULLY")
print("============================================================")
print("VALID_EVENTS is now available.")
print("Proceed to SECTION 31B only.")
print("============================================================")

SECTION 31A — EVENT CATALOGUE + S1 VALIDATION
✓ Earth Engine available
✓ STUDY_ROI detected
✓ Event catalogue created: 8 events
Checking: EVENT_001
Checking: EVENT_002
Checking: EVENT_003
Checking: EVENT_004
Checking: EVENT_005
Checking: EVENT_006
Checking: EVENT_007
Checking: EVENT_008_AUG2026


,event_id,pre_start,pre_end,post_start,post_end,pre_images,post_images
0,EVENT_001,2015-06-01,2015-06-18,2015-06-22,2015-07-02,7,6
1,EVENT_002,2015-06-20,2015-07-02,2015-07-04,2015-07-15,6,3
2,EVENT_003,2016-06-01,2016-06-20,2016-06-23,2016-07-05,6,5
3,EVENT_004,2017-08-25,2017-09-05,2017-09-08,2017-09-20,10,10
4,EVENT_005,2018-07-10,2018-07-23,2018-07-26,2018-08-05,11,6
5,EVENT_006,2022-09-01,2022-09-17,2022-09-20,2022-10-02,13,8
6,EVENT_007,2023-07-05,2023-07-20,2023-07-23,2023-08-05,14,16
7,EVENT_008_AUG2026,2026-07-20,2026-08-03,2026-08-04,2026-08-10,23,7



SECTION 31A RESULT
Catalogue events: 8
Valid completed events: 7
✓ EVENT_001 | 2015-06-01 → 2015-06-18 | 2015-06-22 → 2015-07-02
✓ EVENT_002 | 2015-06-20 → 2015-07-02 | 2015-07-04 → 2015-07-15
✓ EVENT_003 | 2016-06-01 → 2016-06-20 | 2016-06-23 → 2016-07-05
✓ EVENT_004 | 2017-08-25 → 2017-09-05 | 2017-09-08 → 2017-09-20
✓ EVENT_005 | 2018-07-10 → 2018-07-23 | 2018-07-26 → 2018-08-05
✓ EVENT_006 | 2022-09-01 → 2022-09-17 | 2022-09-20 → 2022-10-02
✓ EVENT_007 | 2023-07-05 → 2023-07-20 | 2023-07-23 → 2023-08-05

SECTION 31A COMPLETED SUCCESSFULLY
VALID_EVENTS is now available.
Proceed to SECTION 31B only.


In [28]:
# ============================================================
# SECTION 31B-1 — VERIFY FLOOD-PROCESSING INPUTS
# ============================================================

print("============================================================")
print("SECTION 31B-1 — INPUT DIAGNOSTICS")
print("============================================================")

# Earth Engine
ee.Number(1).getInfo()
print("✓ Earth Engine connection: OK")

# Study ROI
if "STUDY_ROI" not in globals():
    raise NameError("STUDY_ROI is missing.")

print("✓ STUDY_ROI exists")

# Valid events
if "VALID_EVENTS" not in globals():
    raise NameError(
        "VALID_EVENTS is missing. "
        "Run Section 31A first."
    )

print(
    "✓ VALID_EVENTS exists:",
    len(VALID_EVENTS),
    "events"
)

# Permanent water
if "PermanentWater" not in globals():
    raise NameError(
        "PermanentWater is missing."
    )

print(
    "✓ PermanentWater exists"
)

print(
    "✓ PermanentWater bands:",
    PermanentWater.bandNames().getInfo()
)

# Display events
print("\nVALID EVENTS:")
for e in VALID_EVENTS:
    print(
        e["event_id"],
        "|",
        e["pre_start"],
        "→",
        e["pre_end"],
        "|",
        e["post_start"],
        "→",
        e["post_end"]
    )

print("\n✓ SECTION 31B-1 PASSED")

SECTION 31B-1 — INPUT DIAGNOSTICS
✓ Earth Engine connection: OK
✓ STUDY_ROI exists
✓ VALID_EVENTS exists: 7 events
✓ PermanentWater exists
✓ PermanentWater bands: ['PermanentWater']

VALID EVENTS:
EVENT_001 | 2015-06-01 → 2015-06-18 | 2015-06-22 → 2015-07-02
EVENT_002 | 2015-06-20 → 2015-07-02 | 2015-07-04 → 2015-07-15
EVENT_003 | 2016-06-01 → 2016-06-20 | 2016-06-23 → 2016-07-05
EVENT_004 | 2017-08-25 → 2017-09-05 | 2017-09-08 → 2017-09-20
EVENT_005 | 2018-07-10 → 2018-07-23 | 2018-07-26 → 2018-08-05
EVENT_006 | 2022-09-01 → 2022-09-17 | 2022-09-20 → 2022-10-02
EVENT_007 | 2023-07-05 → 2023-07-20 | 2023-07-23 → 2023-08-05

✓ SECTION 31B-1 PASSED


In [29]:
# ============================================================
# SECTION 31B-2 — SINGLE SENTINEL-1 EVENT TEST
# ============================================================

print("============================================================")
print("SECTION 31B-2 — SINGLE EVENT TEST")
print("============================================================")

if "VALID_EVENTS" not in globals():
    raise NameError(
        "VALID_EVENTS is missing. Run Section 31A."
    )

if len(VALID_EVENTS) == 0:
    raise RuntimeError(
        "VALID_EVENTS contains no events."
    )

TEST_EVENT = VALID_EVENTS[0]

print(
    "Testing:",
    TEST_EVENT["event_id"]
)

print(
    "PRE:",
    TEST_EVENT["pre_start"],
    "→",
    TEST_EVENT["pre_end"]
)

print(
    "POST:",
    TEST_EVENT["post_start"],
    "→",
    TEST_EVENT["post_end"]
)


# ------------------------------------------------------------
# Sentinel-1 collection
# ------------------------------------------------------------

def S1_TEST_COLLECTION(start_date, end_date):

    return (
        ee.ImageCollection(
            "COPERNICUS/S1_GRD"
        )
        .filterBounds(STUDY_ROI)
        .filterDate(
            start_date,
            end_date
        )
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH"
            )
        )
        .select(
            ["VV", "VH"]
        )
    )


PRE_TEST = S1_TEST_COLLECTION(
    TEST_EVENT["pre_start"],
    TEST_EVENT["pre_end"]
)

POST_TEST = S1_TEST_COLLECTION(
    TEST_EVENT["post_start"],
    TEST_EVENT["post_end"]
)


pre_count = PRE_TEST.size().getInfo()
post_count = POST_TEST.size().getInfo()

print("\nPRE images :", pre_count)
print("POST images:", post_count)


if pre_count == 0:
    raise RuntimeError(
        "No Sentinel-1 PRE-event images were found."
    )

if post_count == 0:
    raise RuntimeError(
        "No Sentinel-1 POST-event images were found."
    )


# ------------------------------------------------------------
# Create composites
# ------------------------------------------------------------

PRE_IMAGE = PRE_TEST.median()
POST_IMAGE = POST_TEST.median()

print(
    "\nPRE bands:",
    PRE_IMAGE.bandNames().getInfo()
)

print(
    "POST bands:",
    POST_IMAGE.bandNames().getInfo()
)


# ------------------------------------------------------------
# Calculate changes
# ------------------------------------------------------------

VV_PRE_TEST = PRE_IMAGE.select("VV")
VV_POST_TEST = POST_IMAGE.select("VV")

VH_PRE_TEST = PRE_IMAGE.select("VH")
VH_POST_TEST = POST_IMAGE.select("VH")

VV_CHANGE_TEST = (
    VV_POST_TEST
    .subtract(VV_PRE_TEST)
    .rename("VV_change")
)

VH_CHANGE_TEST = (
    VH_POST_TEST
    .subtract(VH_PRE_TEST)
    .rename("VH_change")
)


print(
    "\n✓ VV change created:",
    VV_CHANGE_TEST.bandNames().getInfo()
)

print(
    "✓ VH change created:",
    VH_CHANGE_TEST.bandNames().getInfo()
)

print("\n✓ SECTION 31B-2 PASSED")

SECTION 31B-2 — SINGLE EVENT TEST
Testing: EVENT_001
PRE: 2015-06-01 → 2015-06-18
POST: 2015-06-22 → 2015-07-02

PRE images : 7
POST images: 6

PRE bands: ['VV', 'VH']
POST bands: ['VV', 'VH']

✓ VV change created: ['VV_change']
✓ VH change created: ['VH_change']

✓ SECTION 31B-2 PASSED


In [30]:
# ============================================================
# SECTION 31B-3 — SINGLE EVENT FLOOD MASK
# ============================================================

print("============================================================")
print("SECTION 31B-3 — FLOOD MASK TEST")
print("============================================================")


# ------------------------------------------------------------
# Flood candidate
# ------------------------------------------------------------

FLOOD_CANDIDATE_TEST = (
    VV_CHANGE_TEST
    .lt(-1.5)
    .And(
        VH_CHANGE_TEST.lt(-1.5)
    )
)

print("✓ Sentinel-1 flood candidate created")


# ------------------------------------------------------------
# Remove permanent water
# ------------------------------------------------------------

FLOOD_NONPERMANENT_TEST = (
    FLOOD_CANDIDATE_TEST
    .And(
        PermanentWater.Not()
    )
)

print(
    "✓ Permanent water exclusion applied"
)


# ------------------------------------------------------------
# Remove isolated pixels
# ------------------------------------------------------------

FLOOD_CONNECTED_TEST = (
    FLOOD_NONPERMANENT_TEST
    .selfMask()
    .connectedPixelCount(
        maxSize=100,
        eightConnected=True
    )
    .gte(8)
)


# ------------------------------------------------------------
# Final flood mask
# ------------------------------------------------------------

Flood_TEST = (
    FLOOD_CONNECTED_TEST
    .unmask(0)
    .toByte()
    .rename("Flood")
)

print(
    "✓ Flood mask created"
)

print(
    "Flood bands:",
    Flood_TEST.bandNames().getInfo()
)


# ------------------------------------------------------------
# Test statistics
# ------------------------------------------------------------

flood_pixels = (
    Flood_TEST
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=STUDY_ROI,
        scale=30,
        maxPixels=1e9,
        bestEffort=True
    )
    .get("Flood")
    .getInfo()
)

print(
    "Flood pixels detected:",
    flood_pixels
)


# ------------------------------------------------------------
# Map visualization
# ------------------------------------------------------------

Map.addLayer(
    Flood_TEST.selfMask(),
    {
        "palette": ["red"]
    },
    "TEST Flood Mask — " + TEST_EVENT["event_id"]
)

print("\n✓ SECTION 31B-3 PASSED")

SECTION 31B-3 — FLOOD MASK TEST
✓ Sentinel-1 flood candidate created
✓ Permanent water exclusion applied
✓ Flood mask created
Flood bands: ['Flood']
Flood pixels detected: 3114

✓ SECTION 31B-3 PASSED


In [31]:
# ============================================================
# SECTION 31B-4 — PROCESS ALL VALID FLOOD EVENTS
# ============================================================

print("============================================================")
print("SECTION 31B-4 — FULL FLOOD EVENT PROCESSING")
print("============================================================")


FLOOD_EVENT_IMAGES = []


for event in VALID_EVENTS:

    print("\n------------------------------------------------------------")
    print("Processing:", event["event_id"])
    print("------------------------------------------------------------")

    pre_collection = (
        ee.ImageCollection(
            "COPERNICUS/S1_GRD"
        )
        .filterBounds(STUDY_ROI)
        .filterDate(
            event["pre_start"],
            event["pre_end"]
        )
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH"
            )
        )
        .select(["VV", "VH"])
    )


    post_collection = (
        ee.ImageCollection(
            "COPERNICUS/S1_GRD"
        )
        .filterBounds(STUDY_ROI)
        .filterDate(
            event["post_start"],
            event["post_end"]
        )
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH"
            )
        )
        .select(["VV", "VH"])
    )


    pre_count = pre_collection.size().getInfo()
    post_count = post_collection.size().getInfo()

    print(
        "PRE images :",
        pre_count
    )

    print(
        "POST images:",
        post_count
    )


    if pre_count == 0 or post_count == 0:

        print(
            "⚠ Skipping event because imagery is incomplete."
        )

        continue


    # --------------------------------------------------------
    # Median composites
    # --------------------------------------------------------

    pre_image = pre_collection.median()
    post_image = post_collection.median()


    # --------------------------------------------------------
    # Backscatter
    # --------------------------------------------------------

    VV_pre = (
        pre_image
        .select("VV")
        .rename("VV_pre")
    )

    VV_post = (
        post_image
        .select("VV")
        .rename("VV_post")
    )

    VH_pre = (
        pre_image
        .select("VH")
        .rename("VH_pre")
    )

    VH_post = (
        post_image
        .select("VH")
        .rename("VH_post")
    )


    # --------------------------------------------------------
    # Change detection
    # --------------------------------------------------------

    VV_change = (
        VV_post
        .subtract(VV_pre)
        .rename("VV_change")
    )

    VH_change = (
        VH_post
        .subtract(VH_pre)
        .rename("VH_change")
    )


    # --------------------------------------------------------
    # Flood detection
    # --------------------------------------------------------

    flood_candidate = (
        VV_change
        .lt(-1.5)
        .And(
            VH_change.lt(-1.5)
        )
    )


    # --------------------------------------------------------
    # Permanent water exclusion
    # --------------------------------------------------------

    flood_nonpermanent = (
        flood_candidate
        .And(
            PermanentWater.Not()
        )
    )


    # --------------------------------------------------------
    # Spatial filtering
    # --------------------------------------------------------

    flood_connected = (
        flood_nonpermanent
        .selfMask()
        .connectedPixelCount(
            maxSize=100,
            eightConnected=True
        )
        .gte(8)
    )


    FloodMask = (
        flood_connected
        .unmask(0)
        .toByte()
        .rename("Flood")
    )


    # --------------------------------------------------------
    # Event image
    # --------------------------------------------------------

    event_image = (
        ee.Image.cat(
            [
                VV_pre,
                VV_post,
                VH_pre,
                VH_post,
                VV_change,
                VH_change,
                FloodMask
            ]
        )
        .clip(STUDY_ROI)
        .set(
            "event_id",
            event["event_id"]
        )
        .set(
            "pre_start",
            event["pre_start"]
        )
        .set(
            "pre_end",
            event["pre_end"]
        )
        .set(
            "post_start",
            event["post_start"]
        )
        .set(
            "post_end",
            event["post_end"]
        )
    )


    # --------------------------------------------------------
    # Verify image
    # --------------------------------------------------------

    bands = (
        event_image
        .bandNames()
        .getInfo()
    )

    if "Flood" not in bands:

        raise RuntimeError(
            f"{event['event_id']} does not contain Flood band."
        )


    FLOOD_EVENT_IMAGES.append(
        event_image
    )

    print(
        "✓ Event successfully processed:",
        event["event_id"]
    )


# ============================================================
# FINAL RESULT
# ============================================================

print("\n============================================================")
print("SECTION 31B-4 RESULT")
print("============================================================")

print(
    "Successfully processed:",
    len(FLOOD_EVENT_IMAGES),
    "events"
)


if len(FLOOD_EVENT_IMAGES) == 0:

    raise RuntimeError(
        "No flood events were successfully processed."
    )


print(
    "✓ FLOOD_EVENT_IMAGES created successfully."
)

print(
    "✓ Section 31B is COMPLETE."
)

print(
    "\nDo NOT run Section 31C until this cell succeeds."
)

SECTION 31B-4 — FULL FLOOD EVENT PROCESSING

------------------------------------------------------------
Processing: EVENT_001
------------------------------------------------------------
PRE images : 7
POST images: 6
✓ Event successfully processed: EVENT_001

------------------------------------------------------------
Processing: EVENT_002
------------------------------------------------------------
PRE images : 6
POST images: 3
✓ Event successfully processed: EVENT_002

------------------------------------------------------------
Processing: EVENT_003
------------------------------------------------------------
PRE images : 6
POST images: 5
✓ Event successfully processed: EVENT_003

------------------------------------------------------------
Processing: EVENT_004
------------------------------------------------------------
PRE images : 10
POST images: 10
✓ Event successfully processed: EVENT_004

------------------------------------------------------------
Processing: EVENT_005
--

In [32]:
# ============================================================
# SECTION 31C-1 — BUILD FLOOD EVENT COLLECTION
# ============================================================

print("============================================================")
print("SECTION 31C-1 — FLOOD EVENT COLLECTION")
print("============================================================")

# ------------------------------------------------------------
# 1. Verify required inputs
# ------------------------------------------------------------

if "FLOOD_EVENT_IMAGES" not in globals():
    raise NameError(
        "FLOOD_EVENT_IMAGES is not available. "
        "Section 31B must be completed first."
    )

if len(FLOOD_EVENT_IMAGES) == 0:
    raise RuntimeError(
        "FLOOD_EVENT_IMAGES exists but contains zero images."
    )

print(
    "✓ FLOOD_EVENT_IMAGES detected:",
    len(FLOOD_EVENT_IMAGES),
    "images"
)


# ------------------------------------------------------------
# 2. Create Earth Engine ImageCollection
# ------------------------------------------------------------

FLOOD_COLLECTION = ee.ImageCollection.fromImages(
    FLOOD_EVENT_IMAGES
)


# ------------------------------------------------------------
# 3. Verify collection
# ------------------------------------------------------------

collection_size = (
    FLOOD_COLLECTION
    .size()
    .getInfo()
)

print(
    "✓ FLOOD_COLLECTION size:",
    collection_size
)


if collection_size == 0:
    raise RuntimeError(
        "FLOOD_COLLECTION contains zero images."
    )


# ------------------------------------------------------------
# 4. Verify first image
# ------------------------------------------------------------

FIRST_FLOOD_EVENT = ee.Image(
    FLOOD_COLLECTION.first()
)

FIRST_FLOOD_BANDS = (
    FIRST_FLOOD_EVENT
    .bandNames()
    .getInfo()
)

print(
    "✓ First event bands:"
)

for band in FIRST_FLOOD_BANDS:
    print("   ", band)


# ------------------------------------------------------------
# 5. Verify Flood band
# ------------------------------------------------------------

if "Flood" not in FIRST_FLOOD_BANDS:
    raise RuntimeError(
        "The Flood band is missing from FLOOD_COLLECTION."
    )

print(
    "✓ Required Flood band detected."
)


# ------------------------------------------------------------
# 6. Verify event ID
# ------------------------------------------------------------

FIRST_EVENT_ID = (
    FIRST_FLOOD_EVENT
    .get("event_id")
    .getInfo()
)

print(
    "✓ First event ID:",
    FIRST_EVENT_ID
)


print("\n============================================================")
print("✓ SECTION 31C-1 COMPLETED SUCCESSFULLY")
print("============================================================")

print(
    "FLOOD_COLLECTION is ready for Section 31C-2."
)

SECTION 31C-1 — FLOOD EVENT COLLECTION
✓ FLOOD_EVENT_IMAGES detected: 7 images
✓ FLOOD_COLLECTION size: 7
✓ First event bands:
    VV_pre
    VV_post
    VH_pre
    VH_post
    VV_change
    VH_change
    Flood
✓ Required Flood band detected.
✓ First event ID: EVENT_001

✓ SECTION 31C-1 COMPLETED SUCCESSFULLY
FLOOD_COLLECTION is ready for Section 31C-2.


In [34]:
# ============================================================
# SECTION 31C — MEMORY-SAFE MASTER FLOOD INVENTORY
# ============================================================
#
# Purpose:
#   Build the historical flood inventory from the validated
#   Sentinel-1 event masks without exceeding Earth Engine's
#   user-memory limit.
#
# Inputs:
#   FLOOD_COLLECTION
#   PermanentWater
#   STUDY_ROI
#   ANALYSIS_SCALE
#
# Outputs:
#   FloodFrequency
#   FloodInventory
#   RecurrentFlood
#
# ============================================================

print("============================================================")
print("SECTION 31C — MEMORY-SAFE MASTER FLOOD INVENTORY")
print("============================================================")


# ------------------------------------------------------------
# 1. CHECK EARTH ENGINE
# ------------------------------------------------------------

try:
    ee.Number(1).getInfo()
    print("✓ Earth Engine connection: OK")
except Exception as e:
    raise RuntimeError(
        "Earth Engine is not responding. "
        "Check Section 2 before continuing."
    ) from e


# ------------------------------------------------------------
# 2. CHECK STUDY ROI
# ------------------------------------------------------------

if 'STUDY_ROI' not in globals():
    raise NameError(
        "STUDY_ROI is not defined."
    )

print("✓ STUDY_ROI detected.")


# ------------------------------------------------------------
# 3. CHECK ANALYSIS SCALE
# ------------------------------------------------------------

if 'ANALYSIS_SCALE' not in globals():
    ANALYSIS_SCALE = 30

print(
    "✓ Analysis scale:",
    ANALYSIS_SCALE,
    "m"
)


# ------------------------------------------------------------
# 4. CHECK FLOOD COLLECTION
# ------------------------------------------------------------

if 'FLOOD_COLLECTION' not in globals():
    raise NameError(
        "FLOOD_COLLECTION is not defined.\n"
        "Run Section 31B successfully before Section 31C."
    )

print("✓ FLOOD_COLLECTION detected.")


# ------------------------------------------------------------
# 5. CHECK PERMANENT WATER
# ------------------------------------------------------------

if 'PermanentWater' not in globals():

    print(
        "PermanentWater not found."
    )

    print(
        "Reconstructing PermanentWater from JRC Global "
        "Surface Water..."
    )

    GSW_31C = (
        ee.Image(
            'JRC/GSW1_4/GlobalSurfaceWater'
        )
        .select('occurrence')
        .clip(STUDY_ROI)
    )

    PermanentWater = (
        GSW_31C
        .gte(80)
        .rename('PermanentWater')
    )

    print(
        "✓ PermanentWater reconstructed."
    )

else:

    print(
        "✓ PermanentWater detected."
    )


# ------------------------------------------------------------
# 6. CHECK NUMBER OF EVENTS
# ------------------------------------------------------------

event_count_31C = (
    FLOOD_COLLECTION
    .size()
    .getInfo()
)

print(
    "✓ Flood-event images:",
    event_count_31C
)

if event_count_31C == 0:
    raise RuntimeError(
        "FLOOD_COLLECTION contains zero flood events."
    )


# ------------------------------------------------------------
# 7. CHECK FIRST EVENT BAND
# ------------------------------------------------------------

first_event_31C = ee.Image(
    FLOOD_COLLECTION.first()
)

first_event_bands_31C = (
    first_event_31C
    .bandNames()
    .getInfo()
)

print(
    "✓ First event bands:",
    first_event_bands_31C
)

if 'Flood' not in first_event_bands_31C:
    raise RuntimeError(
        "The FLOOD_COLLECTION does not contain "
        "the required 'Flood' band."
    )


# ============================================================
# 8. MEMORY-SAFE FLOOD FREQUENCY
# ============================================================
#
# Instead of carrying all event bands through the computation,
# retain ONLY the Flood band.
#
# This substantially reduces the server-side image graph.
# ============================================================

print(
    "Building memory-efficient flood-frequency image..."
)

FloodOnlyCollection_31C = (
    FLOOD_COLLECTION
    .select('Flood')
)

print(
    "✓ Flood-only collection prepared."
)


# ------------------------------------------------------------
# 9. SUM FLOOD EVENTS
# ------------------------------------------------------------

FloodFrequency = (
    FloodOnlyCollection_31C
    .sum()
    .rename('FloodFrequency')
    .clip(STUDY_ROI)
)

print(
    "✓ FloodFrequency created."
)


# ============================================================
# 10. PERMANENT WATER MASK
# ============================================================

PermanentWaterMask_31C = (
    PermanentWater
    .select('PermanentWater')
    .eq(1)
)

NonPermanentWater_31C = (
    PermanentWaterMask_31C
    .Not()
)


# ============================================================
# 11. MASTER FLOOD INVENTORY
# ============================================================
#
# FloodInventory = flooded in at least one event
#                  AND not permanent water.
# ============================================================

FloodInventory = (
    FloodFrequency
    .gt(0)
    .And(
        NonPermanentWater_31C
    )
    .rename('FloodInventory')
    .clip(STUDY_ROI)
)

print(
    "✓ FloodInventory created."
)


# ============================================================
# 12. RECURRENT FLOOD
# ============================================================
#
# Recurrent flood = flooded during at least two events.
# ============================================================

RecurrentFlood = (
    FloodFrequency
    .gte(2)
    .And(
        NonPermanentWater_31C
    )
    .rename('RecurrentFlood')
    .clip(STUDY_ROI)
)

print(
    "✓ RecurrentFlood created."
)


# ============================================================
# 13. BAND CHECK
# ============================================================

print(
    "FloodFrequency:",
    FloodFrequency.bandNames().getInfo()
)

print(
    "FloodInventory:",
    FloodInventory.bandNames().getInfo()
)

print(
    "RecurrentFlood:",
    RecurrentFlood.bandNames().getInfo()
)


# ============================================================
# 14. MAP DISPLAY
# ============================================================
#
# Do NOT perform large reduceRegion/getInfo operations here.
# Visualization is deferred to Earth Engine.
# ============================================================

Map.addLayer(
    FloodFrequency,
    {
        'min': 0,
        'max': max(event_count_31C, 1),
        'palette': [
            'ffffff',
            'ffff00',
            'ff9900',
            'ff3300',
            '990000'
        ]
    },
    '31C — Flood Frequency',
    False
)


Map.addLayer(
    FloodInventory.selfMask(),
    {
        'palette': [
            'ff0000'
        ]
    },
    '31C — Master Flood Inventory',
    True
)


Map.addLayer(
    RecurrentFlood.selfMask(),
    {
        'palette': [
            '0000ff'
        ]
    },
    '31C — Recurrent Flood',
    False
)


# ============================================================
# 15. FINAL OBJECT EXISTENCE CHECK
# ============================================================

required_outputs_31C = [
    'FloodFrequency',
    'FloodInventory',
    'RecurrentFlood'
]

missing_outputs_31C = [
    name
    for name in required_outputs_31C
    if name not in globals()
]

if missing_outputs_31C:

    raise RuntimeError(
        "Section 31C failed to create:\n"
        + "\n".join(missing_outputs_31C)
    )


# ============================================================
# 16. SUCCESS MESSAGE
# ============================================================

print()
print("============================================================")
print("✓ SECTION 31C COMPLETED SUCCESSFULLY")
print("============================================================")
print("✓ FloodFrequency")
print("✓ FloodInventory")
print("✓ RecurrentFlood")
print("✓ Permanent water excluded")
print("✓ Memory-intensive regional statistics skipped")
print("✓ Flood layers added to Map")
print("============================================================")

SECTION 31C — MEMORY-SAFE MASTER FLOOD INVENTORY
✓ Earth Engine connection: OK
✓ STUDY_ROI detected.
✓ Analysis scale: 30 m
✓ FLOOD_COLLECTION detected.
✓ PermanentWater detected.
✓ Flood-event images: 7
✓ First event bands: ['VV_pre', 'VV_post', 'VH_pre', 'VH_post', 'VV_change', 'VH_change', 'Flood']
Building memory-efficient flood-frequency image...
✓ Flood-only collection prepared.
✓ FloodFrequency created.
✓ FloodInventory created.
✓ RecurrentFlood created.
FloodFrequency: ['FloodFrequency']
FloodInventory: ['FloodInventory']
RecurrentFlood: ['RecurrentFlood']

✓ SECTION 31C COMPLETED SUCCESSFULLY
✓ FloodFrequency
✓ FloodInventory
✓ RecurrentFlood
✓ Permanent water excluded
✓ Memory-intensive regional statistics skipped
✓ Flood layers added to Map


In [35]:
# ============================================================
# SECTION 32 — FLOOD INVENTORY QUALITY CONTROL
# ============================================================
#
# Purpose:
#   Validate the flood inventory produced in Section 31C.
#
# Inputs:
#   FloodFrequency
#   FloodInventory
#   RecurrentFlood
#   PermanentWater
#   STUDY_ROI
#
# Outputs:
#   Validated FloodInventory
#   FloodFrequency
#   RecurrentFlood
#
# No new large raster calculation is performed.
# ============================================================

print("============================================================")
print("SECTION 32 — FLOOD INVENTORY QUALITY CONTROL")
print("============================================================")


# ------------------------------------------------------------
# 1. CHECK REQUIRED OBJECTS
# ------------------------------------------------------------

required_32 = [
    'STUDY_ROI',
    'FloodFrequency',
    'FloodInventory',
    'RecurrentFlood',
    'PermanentWater'
]

missing_32 = [
    name
    for name in required_32
    if name not in globals()
]

if missing_32:
    raise NameError(
        "SECTION 32 cannot run.\n"
        "Missing objects:\n"
        + "\n".join(missing_32)
    )

print("✓ All required objects detected.")


# ------------------------------------------------------------
# 2. CHECK BAND NAMES
# ------------------------------------------------------------

frequency_bands_32 = (
    FloodFrequency
    .bandNames()
    .getInfo()
)

inventory_bands_32 = (
    FloodInventory
    .bandNames()
    .getInfo()
)

recurrent_bands_32 = (
    RecurrentFlood
    .bandNames()
    .getInfo()
)

water_bands_32 = (
    PermanentWater
    .bandNames()
    .getInfo()
)

print(
    "FloodFrequency bands:",
    frequency_bands_32
)

print(
    "FloodInventory bands:",
    inventory_bands_32
)

print(
    "RecurrentFlood bands:",
    recurrent_bands_32
)

print(
    "PermanentWater bands:",
    water_bands_32
)


# ------------------------------------------------------------
# 3. VERIFY REQUIRED BANDS
# ------------------------------------------------------------

if 'FloodFrequency' not in frequency_bands_32:
    raise RuntimeError(
        "FloodFrequency does not contain the required "
        "'FloodFrequency' band."
    )

if 'FloodInventory' not in inventory_bands_32:
    raise RuntimeError(
        "FloodInventory does not contain the required "
        "'FloodInventory' band."
    )

if 'RecurrentFlood' not in recurrent_bands_32:
    raise RuntimeError(
        "RecurrentFlood does not contain the required "
        "'RecurrentFlood' band."
    )

if 'PermanentWater' not in water_bands_32:
    raise RuntimeError(
        "PermanentWater does not contain the required "
        "'PermanentWater' band."
    )

print("✓ Required bands verified.")


# ------------------------------------------------------------
# 4. VERIFY FLOOD FREQUENCY DATA TYPE
# ------------------------------------------------------------

FloodFrequency = (
    FloodFrequency
    .toInt16()
    .rename('FloodFrequency')
    .clip(STUDY_ROI)
)

print("✓ FloodFrequency converted to integer event count.")


# ------------------------------------------------------------
# 5. VERIFY FLOOD INVENTORY
# ------------------------------------------------------------

FloodInventory = (
    FloodInventory
    .gt(0)
    .rename('FloodInventory')
    .clip(STUDY_ROI)
)

print("✓ FloodInventory converted to binary flood/non-flood.")


# ------------------------------------------------------------
# 6. VERIFY RECURRENT FLOOD
# ------------------------------------------------------------

RecurrentFlood = (
    RecurrentFlood
    .gt(0)
    .rename('RecurrentFlood')
    .clip(STUDY_ROI)
)

print("✓ RecurrentFlood converted to binary.")


# ------------------------------------------------------------
# 7. ENSURE PERMANENT WATER IS EXCLUDED
# ------------------------------------------------------------

PermanentWaterMask_32 = (
    PermanentWater
    .eq(1)
)

FloodInventory = (
    FloodInventory
    .And(
        PermanentWaterMask_32.Not()
    )
    .rename('FloodInventory')
    .clip(STUDY_ROI)
)

RecurrentFlood = (
    RecurrentFlood
    .And(
        PermanentWaterMask_32.Not()
    )
    .rename('RecurrentFlood')
    .clip(STUDY_ROI)
)

print(
    "✓ Permanent water excluded from flood targets."
)


# ------------------------------------------------------------
# 8. FINAL BAND VALIDATION
# ------------------------------------------------------------

print()
print("Final bands:")

print(
    "FloodFrequency:",
    FloodFrequency.bandNames().getInfo()
)

print(
    "FloodInventory:",
    FloodInventory.bandNames().getInfo()
)

print(
    "RecurrentFlood:",
    RecurrentFlood.bandNames().getInfo()
)


# ------------------------------------------------------------
# 9. MAP VISUALIZATION
# ------------------------------------------------------------

Map.addLayer(
    FloodInventory.selfMask(),
    {
        'palette': ['ff0000']
    },
    '32 — Validated Flood Inventory',
    True
)

Map.addLayer(
    RecurrentFlood.selfMask(),
    {
        'palette': ['0000ff']
    },
    '32 — Recurrent Flood',
    False
)


# ------------------------------------------------------------
# 10. SUCCESS CHECK
# ------------------------------------------------------------

print()
print("============================================================")
print("✓ SECTION 32 COMPLETED SUCCESSFULLY")
print("============================================================")
print("✓ FloodFrequency validated")
print("✓ FloodInventory validated")
print("✓ RecurrentFlood validated")
print("✓ Permanent water excluded")
print("✓ No expensive regional statistics performed")
print("============================================================")

SECTION 32 — FLOOD INVENTORY QUALITY CONTROL
✓ All required objects detected.
FloodFrequency bands: ['FloodFrequency']
FloodInventory bands: ['FloodInventory']
RecurrentFlood bands: ['RecurrentFlood']
PermanentWater bands: ['PermanentWater']
✓ Required bands verified.
✓ FloodFrequency converted to integer event count.
✓ FloodInventory converted to binary flood/non-flood.
✓ RecurrentFlood converted to binary.
✓ Permanent water excluded from flood targets.

Final bands:
FloodFrequency: ['FloodFrequency']
FloodInventory: ['FloodInventory']
RecurrentFlood: ['RecurrentFlood']

✓ SECTION 32 COMPLETED SUCCESSFULLY
✓ FloodFrequency validated
✓ FloodInventory validated
✓ RecurrentFlood validated
✓ Permanent water excluded
✓ No expensive regional statistics performed


In [38]:
# ============================================================
# SECTION 33A — FLOOD/NON-FLOOD SAMPLING LOCATIONS
# ============================================================
#
# IMPORTANT:
# This section deliberately uses ONLY FloodInventory.
#
# It does NOT sample MASTER_PREDICTOR_STACK.
# This prevents the Earth Engine memory problem caused by
# evaluating the entire predictor graph during stratifiedSample.
#
# OUTPUT:
#   SAMPLE_POINTS
#
# Classes:
#   0 = non-flood
#   1 = flood
#
# ============================================================

print("============================================================")
print("SECTION 33A — FLOOD/NON-FLOOD SAMPLING LOCATIONS")
print("============================================================")


# ------------------------------------------------------------
# 1. CHECK REQUIRED OBJECTS
# ------------------------------------------------------------

required_33A = [
    'ee',
    'STUDY_ROI',
    'ANALYSIS_SCALE',
    'FloodInventory'
]

# Check Python globals except ee, which is imported as a module.
if 'STUDY_ROI' not in globals():
    raise NameError(
        "STUDY_ROI is not defined. "
        "Run the study-region section first."
    )

if 'ANALYSIS_SCALE' not in globals():
    raise NameError(
        "ANALYSIS_SCALE is not defined."
    )

if 'FloodInventory' not in globals():
    raise NameError(
        "FloodInventory is not defined. "
        "Run Sections 31C and 32 successfully first."
    )

print("✓ STUDY_ROI detected.")
print("✓ ANALYSIS_SCALE detected.")
print("✓ FloodInventory detected.")


# ------------------------------------------------------------
# 2. VERIFY FLOOD INVENTORY BAND
# ------------------------------------------------------------

flood_bands_33A = (
    FloodInventory
    .bandNames()
    .getInfo()
)

print(
    "FloodInventory bands:",
    flood_bands_33A
)

if 'FloodInventory' not in flood_bands_33A:
    raise RuntimeError(
        "FloodInventory does not contain the required "
        "'FloodInventory' band."
    )

print("✓ FloodInventory band verified.")


# ------------------------------------------------------------
# 3. CREATE CLEAN BINARY TARGET
# ------------------------------------------------------------
#
# 0 = non-flood
# 1 = flood
#
# unmask(0) is intentional here because the sampling target
# must contain both classes explicitly.
# ------------------------------------------------------------

FloodTarget_33A = (
    FloodInventory
    .gt(0)
    .unmask(0)
    .toByte()
    .rename('FloodTarget')
    .clip(STUDY_ROI)
)

print("✓ Binary FloodTarget created.")


# ------------------------------------------------------------
# 4. CHECK TARGET BAND
# ------------------------------------------------------------

print(
    "FloodTarget bands:",
    FloodTarget_33A
    .bandNames()
    .getInfo()
)


# ------------------------------------------------------------
# 5. CREATE STRATIFIED SAMPLE POINTS
# ------------------------------------------------------------
#
# IMPORTANT:
# Only FloodTarget is passed to stratifiedSample().
#
# Therefore Earth Engine does NOT have to evaluate the large
# MASTER_PREDICTOR_STACK at this stage.
#
# We initially use 1,500 points per class.
# This gives up to 3,000 points and is deliberately conservative
# while we stabilize the workflow.
#
# If this succeeds, Section 33B can sample the predictors at
# these already-created locations.
# ------------------------------------------------------------

print()
print(
    "Creating flood/non-flood sampling locations..."
)

SAMPLE_POINTS = (
    FloodTarget_33A
    .stratifiedSample(
        numPoints=3000,

        classBand='FloodTarget',

        region=STUDY_ROI,

        scale=ANALYSIS_SCALE,

        classValues=[
            0,
            1
        ],

        classPoints=[
            1500,
            1500
        ],

        seed=2026,

        geometries=True,

        dropNulls=False,

        tileScale=16
    )
)

print(
    "✓ Sampling operation submitted."
)


# ------------------------------------------------------------
# 6. COUNT SAMPLE POINTS
# ------------------------------------------------------------

sample_points_count_33A = (
    SAMPLE_POINTS
    .size()
    .getInfo()
)

print()
print(
    "Number of sample points:",
    sample_points_count_33A
)

if sample_points_count_33A == 0:
    raise RuntimeError(
        "No sampling points were generated."
    )


# ------------------------------------------------------------
# 7. CHECK AVAILABLE CLASSES
# ------------------------------------------------------------

sample_classes_33A = (
    SAMPLE_POINTS
    .aggregate_array(
        'FloodTarget'
    )
    .distinct()
    .sort()
    .getInfo()
)

print(
    "Classes detected:",
    sample_classes_33A
)

if 0 not in sample_classes_33A:
    raise RuntimeError(
        "No non-flood sampling points were generated."
    )

if 1 not in sample_classes_33A:
    raise RuntimeError(
        "No flood sampling points were generated."
    )


# ------------------------------------------------------------
# 8. COUNT EACH CLASS
# ------------------------------------------------------------

non_flood_points_33A = (
    SAMPLE_POINTS
    .filter(
        ee.Filter.eq(
            'FloodTarget',
            0
        )
    )
    .size()
    .getInfo()
)

flood_points_33A = (
    SAMPLE_POINTS
    .filter(
        ee.Filter.eq(
            'FloodTarget',
            1
        )
    )
    .size()
    .getInfo()
)

print()
print(
    "Non-flood points:",
    non_flood_points_33A
)

print(
    "Flood points:",
    flood_points_33A
)


# ------------------------------------------------------------
# 9. MAP SAMPLE LOCATIONS
# ------------------------------------------------------------

Map.addLayer(
    FloodTarget_33A.selfMask(),
    {
        'palette': [
            'ff0000'
        ]
    },
    '33A — Flood Target',
    False
)

Map.addLayer(
    SAMPLE_POINTS,
    {
        'color': '00ffff'
    },
    '33A — Sampling Points',
    True
)


# ------------------------------------------------------------
# 10. FINAL SUCCESS CHECK
# ------------------------------------------------------------

if (
    non_flood_points_33A == 0
    or
    flood_points_33A == 0
):
    raise RuntimeError(
        "Sampling failed to produce both flood and "
        "non-flood classes."
    )


print()
print("============================================================")
print("✓ SECTION 33A COMPLETED SUCCESSFULLY")
print("============================================================")
print(
    "Total sampling points:",
    sample_points_count_33A
)
print(
    "Non-flood points:",
    non_flood_points_33A
)
print(
    "Flood points:",
    flood_points_33A
)
print()
print("✓ SAMPLE_POINTS is now available.")
print("✓ MASTER_PREDICTOR_STACK was NOT evaluated.")
print("✓ The memory-intensive operation has been avoided.")
print("============================================================")

SECTION 33A — FLOOD/NON-FLOOD SAMPLING LOCATIONS
✓ STUDY_ROI detected.
✓ ANALYSIS_SCALE detected.
✓ FloodInventory detected.
FloodInventory bands: ['FloodInventory']
✓ FloodInventory band verified.
✓ Binary FloodTarget created.
FloodTarget bands: ['FloodTarget']

Creating flood/non-flood sampling locations...
✓ Sampling operation submitted.


EEException: Computation timed out.